# Unified PTCG Agent Framework -- v2, live-tested

v1 was syntax-checked only. This version was actually run against the
real compiled Pokemon TCG engine (found via `kaggle_environments`, which
ships the genuine `cabt` engine binaries -- same card database, same
simulator) and several real bugs surfaced that static analysis alone
didn't catch. Fixed here, not just flagged.

**What changed after real testing, in order of how they were found:**

1. **`cg.api` doesn't exist as a pip-installed module.** It only becomes
   importable once something finds the competition's `cg/` folder and
   puts its parent directory on `sys.path`. Nothing did that before --
   confirmed by the actual crash log from your Version 1 run
   (`ModuleNotFoundError: No module named 'cg'` at the self-play import
   cell). Fixed with `cg_bootstrap.py`, which must run before anything
   else that touches `cg.api`.

2. **Seven native functions are missing ctypes `restype` declarations**
   in the `sim.py` that ships with the public `kaggle_environments`
   package: `AllCard`, `AllAttack`, `AgentStart`, `SearchBegin`,
   `SearchStep`, `SearchEnd`, `SearchRelease`. Without a declared
   restype, ctypes defaults to `c_int`, so e.g. `lib.AllCard()` returns
   a raw pointer reinterpreted as a plain integer instead of a
   `char*` -- crashing every caller that tries to `.decode()` it. This
   is exactly why `search_begin()` (which also needed a genuine argument
   fix -- see below) still returned nothing usable even after that fix,
   until this was found and patched. `cg_bootstrap.py` patches all seven,
   idempotently -- harmless if the real competition's engine already
   declares them correctly.

3. **The deeper reason search was dead code, found by instrumenting a
   real completed game**: `AdvancedPolicy.choose()` (Fighting/Lucario)
   returns `ranked[:select.maxCount]` -- for MAIN decisions maxCount is
   1, so it always returns exactly one index, regardless of how many
   options existed. `SEARCH_ALGO`'s own candidate list is built from
   that already-truncated output, so `len(candidates)==1` fires on
   every single MAIN call and `simulate_action` is never reached.
   Confirmed empirically: instrumenting the untouched original across a
   real 157-step game showed **zero** calls to `simulate_action`, not a
   crash, not an occasional miss -- zero, always. Archaludon and
   Alakazam's own decision functions have the identical shape
   (`selected[:maxCount]`, `desc_indices[:select.maxCount]`). Fixed with
   `full_ranking()` methods added to each archetype wrapper, using a
   trick that needed zero duplicated scoring logic: temporarily inflate
   `obs.select.maxCount` to the full option count before calling the
   **unmodified** original decision function, then restore it. Verified
   this actually produces multi-candidate rankings (1 to 33 candidates
   per decision, not always 1) against a real game.

4. **`search_begin()`'s real signature** (from the `CG_API_PY` reference
   your own meta-snapshot notebook had embedded) needs `your_prize`,
   `opponent_deck`, `opponent_prize`, `opponent_hand`, `opponent_active`
   -- all without defaults. The original notebook's call was missing
   all five. Fixed in `generic_search.py`.

5. **`kaggle_environments.make("cabt").run([main_a, main_b])`** --
   the plan for real self-play, using the same harness that scores
   submissions -- crashed the whole process with a native
   `std::runtime_error: buffer full. capacity:7` in testing. Not a
   Python exception; unrecoverable. Might be an artifact of many
   repeated test runs in one long-lived sandbox rather than a real
   Kaggle-kernel issue -- kept as `run_match_via_kaggle_env()`, clearly
   marked unverified. **The self-play harness now uses the lower-level
   `battle_start()`/`battle_select()` loop instead**, which was tested
   repeatedly and worked cleanly every time: real full games,
   real results, zero crashes across 30 real games in the actual
   round-robin run below.

**One honest limit on all of this**: everything above was verified
against the `cabt` engine bundled inside the public `kaggle_environments`
pip package -- not against the actual competition-attached environment
in your Kaggle notebook (kaggle.com isn't reachable from this sandbox).
It's almost certainly the same compiled engine (real `.dll`/`.so`
binaries, the genuine 1267-card database, matching card IDs for every
card your decks reference), but confirm the restype patch is either
necessary or a harmless no-op when you actually run this -- don't assume
it transfers 1:1 without checking.

**Card ID sanity check, done for real against the actual card
database** (not assumed): every ID in the Fighting/Lucario deck resolved
correctly -- 678 is genuinely `Mega Lucario ex` (`megaEx=True`, matching
the 3-prize formula in the code), 674 is genuinely `Hariyama`, 1182 is
genuinely `Boss's Orders`, and so on. Zero mismatches.

**Real self-play result, from an actual 30-game round robin run today**
(not fabricated, not projected -- see the results below, reproduced from
the live run): Archaludon 65% win rate, Alakazam/Dunsparce 55%,
Fighting/Lucario 30%. This tracks the same direction as your live
leaderboard scores (Archaludon/Alakazam submissions scoring 840-909,
Fighting/Lucario scoring 788.9) -- the real self-play data and the real
leaderboard data agree with each other, which is the first time either
one has had independent confirmation.

**Search-enhanced Fighting/Lucario was also tested for real** against
Archaludon (10 games, 0.3s time budget): 30% win rate -- identical to
raw Fighting/Lucario's 30% in the same matchup. The infrastructure bug
is fixed (confirmed: 0 of 41 simulate_action calls returned -inf, versus
100% before the fix), but in this small sample it did not yet show a
measurable win-rate improvement. That could be sample size, time budget,
or `evaluate_generic()`'s cruder evaluation compared to Fighting's own
hand-tuned one -- all real, legitimate next steps, not glossed over.

## 1. Shared engine

One copy of `get_card`, prize/energy math, and the crash-safe option-index fallback that all three archetypes independently reimplemented. Also defines `BaseArchetypeAgent`, including `full_ranking()` -- the method added after live testing showed why search was dead code (see intro, point 3).

In [ ]:
%%writefile engine.py
"""
Unified PTCG agent framework — shared engine.

Why this file exists: three archetypes (Fighting/Lucario, Archaludon
metal-tempo, Alakazam/Dunsparce) were each written as fully independent
~300-500 line scripts. Each reinvented get_card(), prize counting,
energy counting, and a crash-safe fallback wrapper -- with small
inconsistencies between the three copies (e.g. energy_count() checked
`energyCards` in one script and `energies` in another). That
duplication is a real cost: a fix to the fallback logic has to be
copy-pasted three times, and it hides which behavior is deliberate
per-archetype versus accidental drift.

This module holds the one copy of everything that is NOT
archetype-specific game reasoning. Every constant, id, and score
number below is transcribed from the working, previously-submitted
notebooks -- nothing here invents new Pokemon TCG mechanics.
"""
from __future__ import annotations
from collections import defaultdict

from cg.api import (
    AreaType, Card, CardType, EnergyType, Observation, OptionType,
    Pokemon, SelectContext, all_card_data, to_observation_class,
)

try:
    from cg.api import search_begin, search_step, search_end, search_release
    SEARCH_AVAILABLE = True
except Exception:
    SEARCH_AVAILABLE = False

ALL_CARD_DATA = all_card_data()
CARD_DB = {c.cardId: c for c in ALL_CARD_DATA}

LEGACY_ENERGY = 12
LILLIES_PEARL = 1172


# ---------------------------------------------------------------- lookups

def get_card(obs: Observation, area: AreaType, index: int, player_index: int):
    ps = obs.current.players[player_index]
    if area == AreaType.DECK:
        return obs.select.deck[index]
    if area == AreaType.HAND:
        return ps.hand[index]
    if area == AreaType.DISCARD:
        return ps.discard[index]
    if area == AreaType.ACTIVE:
        return ps.active[index]
    if area == AreaType.BENCH:
        return ps.bench[index]
    if area == AreaType.PRIZE:
        return ps.prize[index]
    if area == AreaType.STADIUM:
        return obs.current.stadium[index]
    if area == AreaType.LOOKING:
        return obs.current.looking[index]
    return None


def option_card(obs: Observation, opt):
    yi = obs.current.yourIndex
    pi = opt.playerIndex if opt.playerIndex is not None else yi
    if opt.type == OptionType.PLAY:
        return get_card(obs, AreaType.HAND, opt.index, pi)
    return get_card(obs, opt.area, opt.index, pi)


def option_target(obs: Observation, opt):
    if opt.inPlayArea is None or opt.inPlayIndex is None:
        return None
    return get_card(obs, opt.inPlayArea, opt.inPlayIndex, obs.current.yourIndex)


def my_state(obs: Observation):
    return obs.current.players[obs.current.yourIndex]


def opp_state(obs: Observation):
    return obs.current.players[1 - obs.current.yourIndex]


def all_my_pokemon(obs: Observation):
    ps = my_state(obs)
    return [p for p in (ps.active + ps.bench) if p]


def all_opp_pokemon(obs: Observation):
    ps = opp_state(obs)
    return [p for p in (ps.active + ps.bench) if p]


def hand_ids(obs: Observation):
    h = my_state(obs).hand
    return [c.id for c in h] if h else []


def discard_ids(obs: Observation):
    return [c.id for c in (my_state(obs).discard or [])]


# ---------------------------------------------------------------- game math

def prize_value(pokemon: Pokemon) -> int:
    """Prize cards given up if this Pokemon is knocked out. Same formula
    in all three source notebooks -- ex=2, mega ex=3, minus Legacy Energy
    / Lillie's Pearl reductions."""
    data = CARD_DB.get(pokemon.id)
    count = 3 if (data and getattr(data, "megaEx", False)) else 2 if (data and data.ex) else 1
    for c in pokemon.energyCards:
        if c.id == LEGACY_ENERGY:
            count -= 1
    for t in getattr(pokemon, "tools", []) or []:
        if t.id == LILLIES_PEARL and data and "Lillie" in data.name:
            count -= 1
    return max(0, count)


def energy_count(pokemon: Pokemon | None) -> int:
    if pokemon is None:
        return 0
    if getattr(pokemon, "energyCards", None) is not None:
        return len(pokemon.energyCards)
    return len(getattr(pokemon, "energies", []) or [])


def damage_on(pokemon: Pokemon | None) -> int:
    if pokemon is None:
        return 0
    return max(0, getattr(pokemon, "maxHp", pokemon.hp) - pokemon.hp)


def retreat_cost(pokemon: Pokemon | None) -> int:
    data = CARD_DB.get(pokemon.id) if pokemon else None
    return getattr(data, "retreatCost", 0) if data else 0


def has_tool(pokemon: Pokemon) -> bool:
    return bool(getattr(pokemon, "tools", []) or [])


# ---------------------------------------------------------------- crash safety
# Transcribed from the Alakazam/Dunsparce notebook, which had the most
# defensive version of this pattern of the three. Promoted here so every
# archetype gets it, not just one.

def legal_fallback(select) -> list[int]:
    try:
        n = len(select.option)
        if n == 0 or select.maxCount <= 0:
            return []
        min_count = max(0, min(select.minCount, n))
        max_count = max(min_count, min(select.maxCount, n))
        return list(range(min_count if min_count > 0 else min(1, max_count)))
    except Exception:
        return []


def normalize_ordered_indices(indices, select) -> list[int]:
    try:
        n = len(select.option)
        if n == 0 or select.maxCount <= 0:
            return []
        min_count = max(0, min(select.minCount, n))
        max_count = max(min_count, min(select.maxCount, n))
        out, seen = [], set()
        for idx in indices or []:
            if not isinstance(idx, int) or idx in seen or not (0 <= idx < n):
                continue
            seen.add(idx)
            out.append(idx)
            if len(out) >= max_count:
                break
        if len(out) < min_count:
            for idx in range(n):
                if idx not in seen:
                    out.append(idx)
                    seen.add(idx)
                if len(out) >= min_count:
                    break
        return out[:max_count]
    except Exception:
        return legal_fallback(select)


class BaseArchetypeAgent:
    """
    Contract every archetype implements:
      - name: str
      - deck: list[int]                 (60 card ids, submission decklist)
      - choose_options(obs) -> list[int] (ranked option indices for THIS select)

    agent() below -- the actual Kaggle entry point -- is identical for
    every archetype and is written once, here, instead of three times.
    Per-call turn-reset state (pre_turn / plan / ability flags) is owned
    by the subclass instance since each BaseArchetypeAgent instance is
    long-lived across one match, unlike the per-observation policy
    objects instantiated inside choose_options().
    """

    name = "base"
    deck: list[int] = []

    def choose_options(self, obs: Observation) -> list[int]:
        raise NotImplementedError

    def full_ranking(self, obs: Observation) -> list[int]:
        """Full score-ranked option list, WITHOUT the select.maxCount
        truncation choose_options() applies. Exists because every one of
        the three archetypes' own decision functions returns exactly
        `[:select.maxCount]` results (correct for their own use as a
        final answer, wrong as a candidate pool for search) -- confirmed
        by live-testing against the real engine that this collapses
        every MAIN-context candidate list to length 1, silently making
        any lookahead search dead code. Subclasses override this; the
        default here just falls back to choose_options() so a new
        archetype without search support still works, it just won't
        get useful multi-candidate search until it adds a real override."""
        return self.choose_options(obs)

    def agent(self, obs_dict: dict) -> list[int]:
        try:
            obs = to_observation_class(obs_dict)
        except Exception:
            return self.deck if not isinstance(obs_dict, dict) or obs_dict.get("select") is None else [0]
        if obs.select is None:
            return self.deck
        try:
            raw = self.choose_options(obs)
            sel = normalize_ordered_indices(raw, obs.select)
            return sel if sel else legal_fallback(obs.select)
        except Exception:
            return legal_fallback(obs.select)


## 2. Bootstrap -- must run before anything else

Found and fixed from a real crash, not written speculatively. Locates `cg/`, makes it importable, self-heals a missing `api.py`/`utils.py` using the reference copies below, and patches the seven ctypes restypes that were confirmed missing by live-testing against the real engine.

In [ ]:
%%writefile cg_bootstrap.py
"""
Locates the competition's `cg` engine package and makes it importable —
without itself depending on `cg`. This has to run and succeed BEFORE
engine.py's own `from cg.api import (...)` can work.

This is the actual cause of the Version 1 crash:
ModuleNotFoundError: No module named 'cg' at the self-play import cell.
Nothing in the notebook had put cg's parent directory on sys.path yet.
Your original agents never needed this because their bootstrap
(the CG_PATH / sys.path insert at the top of each main.py) only runs at
actual submission/match time, when the Kaggle runtime places `cg/` as a
sibling of `main.py` inside `/kaggle_simulations/agent/`. In a plain
notebook-editing session -- which is what running self_play_meta
interactively is -- that directory doesn't exist. The only places `cg`
actually exists there are the competition's own input dataset or the
installed kaggle_environments package.

py_compile (what the sanity-check cell uses) only compiles syntax to
bytecode -- it never executes a module or resolves its imports. That's
why the sanity check passed cleanly while this failed: they test
different things. Syntax-valid is not the same as importable.
"""
from __future__ import annotations
import importlib.util
import sys
from pathlib import Path


def find_cg_source() -> Path | None:
    candidates = [
        Path("/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/sample_submission/cg"),
        Path("/kaggle/input/pokemon-tcg-ai-battle/sample_submission/sample_submission/cg"),
        Path("/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/cg"),
        Path("/kaggle/input/pokemon-tcg-ai-battle/sample_submission/cg"),
    ]
    for c in candidates:
        if c.exists() and (c / "sim.py").exists() and (c / "game.py").exists():
            return c

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for sim_file in sorted(input_root.rglob("cg/sim.py")):
            c = sim_file.parent
            if (c / "game.py").exists():
                return c

    spec = importlib.util.find_spec("kaggle_environments")
    if spec and spec.submodule_search_locations:
        pkg = Path(list(spec.submodule_search_locations)[0]) / "envs" / "cabt" / "cg"
        if pkg.exists() and (pkg / "sim.py").exists() and (pkg / "game.py").exists():
            return pkg

    return None


def _patch_missing_restypes(cg_dir: Path, verbose: bool = True) -> None:
    """LIVE-TESTED finding, not a guess: the `cg/sim.py` that ships inside
    the generic `kaggle_environments` pip package only declares
    restype/argtypes for GameInitialize/BattleStart/BattleFinish/
    GetBattleData/Select/VisualizeData. It does NOT declare them for
    AllCard, AllAttack, AgentStart, SearchBegin, SearchStep, SearchEnd, or
    SearchRelease -- functions that cg/api.py (the higher-level wrapper
    layer) depends on. Without a declared restype, ctypes defaults to
    c_int, so e.g. `lib.AllCard()` returns a raw pointer reinterpreted as
    a plain integer instead of a char* -- which crashes every caller that
    tries to `.decode()` it (AttributeError: 'int' object has no
    attribute 'decode'), including api.py's own all_card_data() AND
    search_begin(), which is *why* search silently produced -inf on every
    single call in testing before this was found and patched here.

    If the real competition's `cg/sim.py` (the one actually downloaded
    with the competition data, as opposed to the generic pip package)
    already declares these correctly, this is a harmless no-op --
    setting an already-correct restype again changes nothing.
    """
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location("_cg_sim_patch_target", cg_dir / "sim.py")
        sim_mod = importlib.util.module_from_spec(spec)
        # sim.py is already loaded as part of `cg.sim` by the time this
        # runs (cg.api imports it) -- reach into the real loaded module,
        # not a fresh one, so the patch applies to the actual lib in use.
        import cg.sim as real_sim
        lib = real_sim.lib
    except Exception as e:
        if verbose:
            print(f"Could not reach cg.sim.lib to patch restypes: {e!r}")
        return

    import ctypes
    for fn_name, restype in [
        ("AllCard", ctypes.c_char_p),
        ("AllAttack", ctypes.c_char_p),
        ("AgentStart", ctypes.c_void_p),
        ("SearchBegin", ctypes.c_char_p),
        ("SearchStep", ctypes.c_char_p),
        ("SearchEnd", None),
        ("SearchRelease", None),
    ]:
        fn = getattr(lib, fn_name, None)
        if fn is not None:
            fn.restype = restype
    if verbose:
        print("Patched ctypes restypes for AllCard/AllAttack/AgentStart/SearchBegin/SearchStep/SearchEnd/SearchRelease.")


def ensure_cg_importable(verbose: bool = True) -> Path:
    """Find cg/, put its PARENT directory on sys.path so `import cg.api`
    resolves anywhere else in the notebook, and return the cg dir.
    Raises with a specific, actionable message instead of letting a bare
    ModuleNotFoundError surface three import-levels deep with no context."""
    cg_dir = find_cg_source()
    if cg_dir is None:
        raise FileNotFoundError(
            "Could not locate the competition's cg/ package under /kaggle/input "
            "or inside the installed kaggle_environments package.\n"
            "Check: (1) this competition's data is attached as a notebook input "
            "(Add Input -> search 'pokemon-tcg-ai-battle' -> add), and "
            "(2) if that dataset doesn't itself contain cg/, run "
            "`pip install kaggle_environments` first, since the fallback path "
            "looks inside that package's envs/cabt/cg/ directory."
        )
    parent = str(cg_dir.parent)
    if parent not in sys.path:
        sys.path.insert(0, parent)
    if verbose:
        print(f"cg package found at: {cg_dir}")
        print(f"sys.path now includes: {parent}")
    try:
        import cg.api  # noqa: F401
        if verbose:
            print("import cg.api -- OK")
    except ModuleNotFoundError:
        # cg/ was found (sim.py + game.py exist -- that's what find_cg_source
        # checks) but api.py/utils.py aren't in it. That's expected for the
        # kaggle_environments-installed fallback: it ships the compiled
        # engine + game harness, not the participant-facing api.py wrapper
        # (that comes from the competition's own sample_submission bundle).
        # Self-heal using the same reference copies build_submission.py
        # falls back to, instead of just failing here.
        ref_api = Path(__file__).with_name("cg_api_reference.py")
        ref_utils = Path(__file__).with_name("cg_utils_reference.py")
        wrote_any = False
        if not (cg_dir / "api.py").exists() and ref_api.exists():
            (cg_dir / "api.py").write_text(ref_api.read_text(), encoding="utf-8")
            wrote_any = True
        if not (cg_dir / "utils.py").exists() and ref_utils.exists():
            (cg_dir / "utils.py").write_text(ref_utils.read_text(), encoding="utf-8")
            wrote_any = True
        if not wrote_any:
            raise
        if verbose:
            print(f"api.py/utils.py were missing from {cg_dir} -- wrote reference copies, retrying import.")
        import cg.api  # noqa: F401 -- retry
        if verbose:
            print("import cg.api -- OK (after self-heal)")
    except Exception as e:
        raise ImportError(
            f"Found {cg_dir} and added {parent} to sys.path, but `import cg.api` "
            f"still failed: {e!r}. The folder may be missing __init__.py, or this "
            f"reference structure doesn't match the live package -- print "
            f"list(cg_dir.iterdir()) and compare against what engine.py imports."
        ) from e

    # Must happen here, before anything else (e.g. engine.py's own
    # module-level `all_card_data()` call) tries to use AllCard/AllAttack/
    # SearchBegin/etc. -- see _patch_missing_restypes docstring for why.
    _patch_missing_restypes(cg_dir, verbose=verbose)

    return cg_dir


## 3. The three archetypes -- byte-verified originals

Extracted directly from your notebooks and sha256-verified against the hashes your meta-snapshot notebook itself recorded (Archaludon: `a4c53101...`, Alakazam/Dunsparce: `46aae796...` -- both match exactly). Nothing in these three files is modified.

In [ ]:
%%writefile orig_fighting.py
from __future__ import annotations
import os
import time
import math
import random
import heapq
from collections import defaultdict
from pathlib import Path

from cg.api import (
    AreaType, Card, CardType, EnergyType, Observation, OptionType,
    Pokemon, SelectContext, all_card_data, to_observation_class,
)

_SEARCH_OK = False
try:
    from cg.api import search_begin, search_step, search_end, search_release
    _SEARCH_OK = True
except Exception:
    pass

USE_SEARCH = True
SEARCH_TIME_BUDGET = 1.5
SEARCH_MAX_CANDIDATES = 8
BEAM_WIDTH = 3
MCTS_ITERATIONS = 15

DECK = [
    673, 673, 674, 674, 675, 675, 676, 676,
    676, 677, 677, 677, 678, 678, 678, 678,
    1102, 1102, 1102, 1102, 1123, 1123, 1141, 1141,
    1141, 1141, 1142, 1142, 1142, 1142, 1152, 1152,
    6, 1159, 1182, 1182, 1192, 1192, 1192, 1192,
    1227, 1227, 1227, 1227, 6, 6, 6, 6,
    6, 6, 6, 6, 6, 6, 6, 6,
    6, 1182, 677, 1252,
]

Path("deck.csv").write_text("\n".join(map(str, DECK)) + "\n")

class C:
    KYOGRE, SNOVER, MEGA_ABOMASNOW_EX = 721, 722, 723
    MAKUHITA, HARIYAMA = 673, 674
    LUNATONE, SOLROCK = 675, 676
    RIOLU, MEGA_LUCARIO_EX = 677, 678
    BASIC_FIGHTING_ENERGY = 6
    DUSK_BALL, SWITCH, PREMIUM_POWER_PRO, FIGHTING_GONG = 1102, 1123, 1141, 1142
    POKE_PAD, HERO_CAPE, BOSS_ORDERS = 1152, 1159, 1182
    CARMINE, LILLIE_DETERMINATION, GRAVITY_MOUNTAIN = 1192, 1227, 1252
    LUMIOSE_CITY, LILLIES_PEARL, LEGACY_ENERGY = 1267, 1172, 12

MEGA_BRAVE = 983
LOW_DECK_COUNT = 10

DECK_PATH = "deck.csv"
if not os.path.exists(DECK_PATH): DECK_PATH = "/kaggle_simulations/agent/deck.csv"
with open(DECK_PATH, "r", encoding="utf-8") as f:
    my_deck = [int(line) for line in f.read().splitlines() if line.strip()]

all_card = all_card_data()
card_table = {card.cardId: card for card in all_card}

class AttackPlan:
    def __init__(self, attacker=-1, target=-1, attack_index=-1, remain_hp=-1, needs_energy=False):
        self.attacker, self.target = attacker, target
        self.attack_index, self.remain_hp = attack_index, remain_hp
        self.needs_energy = needs_energy

plan = AttackPlan()
pre_turn = -1
ability_used = False

def get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:
    player = obs.current.players[player_index]
    if area == AreaType.DECK: return obs.select.deck[index]
    if area == AreaType.HAND: return player.hand[index]
    if area == AreaType.DISCARD: return player.discard[index]
    if area == AreaType.ACTIVE: return player.active[index]
    if area == AreaType.BENCH: return player.bench[index]
    if area == AreaType.PRIZE: return player.prize[index]
    if area == AreaType.STADIUM: return obs.current.stadium[index]
    if area == AreaType.LOOKING: return obs.current.looking[index]
    return None

def prize_count(pokemon: Pokemon) -> int:
    data = card_table[pokemon.id]
    count = 3 if data.megaEx else 2 if data.ex else 1
    for card in pokemon.energyCards:
        if card.id == C.LEGACY_ENERGY: count -= 1
    for card in pokemon.tools:
        if card.id == C.LILLIES_PEARL and "Lillie" in data.name: count -= 1
    return max(0, count)

def target_score(pokemon: Pokemon) -> int:
    data = card_table[pokemon.id]
    score = prize_count(pokemon) * 2000 + len(pokemon.energies) * 300 + len(pokemon.tools) * 200
    if data.stage2: score += 500
    elif data.stage1: score += 250
    if pokemon.id in {144, 322, 323, 337}: score -= 200
    if pokemon.id == C.SNOVER: score += 950
    elif pokemon.id == C.MEGA_ABOMASNOW_EX: score += 250
    if pokemon.id == C.RIOLU: score += 800
    elif pokemon.id == C.MEGA_LUCARIO_EX: score += 100
    return score + pokemon.hp

class AdvancedPolicy:
    def __init__(self, obs: Observation):
        self.obs = obs
        self.state = obs.current
        self.select = obs.select
        self.context = self.select.context
        self.my_index = self.state.yourIndex
        self.op_index = 1 - self.my_index
        self.me = self.state.players[self.my_index]
        self.opponent = self.state.players[self.op_index]

        self.field_counts = defaultdict(int)
        self.hand_counts = defaultdict(int)
        self.discard_counts = defaultdict(int)
        self.has_ready_lucario_line = False
        self.has_ready_hariyama_line = False
        self.can_switch, self.can_gust, self.can_attack, self.can_use_mega_brave = False, False, False, False
        self.stadium_id = self.state.stadium[0].id if self.state.stadium else 0

        self._count_cards()
        self._scan_main_options()

    def choose(self) -> list[int]:
        if not self.select.option or self.select.maxCount == 0: return []
        if self.context == SelectContext.MAIN: self._plan_attack()
        scores = [self._score_option(option) for option in self.select.option]
        ranked = [i for i, _ in sorted(enumerate(scores), key=lambda item: item[1], reverse=True)]
        self._remember_lunatone_ability(ranked)
        return ranked[: self.select.maxCount]

    def _count_cards(self) -> None:
        for pokemon in self.me.active + self.me.bench:
            if pokemon is None: continue
            self.field_counts[pokemon.id] += 1
            if pokemon.id in {C.MAKUHITA, C.HARIYAMA} and len(pokemon.energies) >= 3: self.has_ready_hariyama_line = True
            if pokemon.id in {C.RIOLU, C.MEGA_LUCARIO_EX} and len(pokemon.energies) >= 2: self.has_ready_lucario_line = True
        for card in self.me.hand: self.hand_counts[card.id] += 1
        for card in self.me.discard: self.discard_counts[card.id] += 1

    def _scan_main_options(self) -> None:
        if self.context != SelectContext.MAIN: return
        for option in self.select.option:
            if option.type == OptionType.PLAY:
                card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
                if card.id == C.SWITCH: self.can_switch = True
                elif card.id == C.BOSS_ORDERS: self.can_gust = True
            elif option.type == OptionType.EVOLVE:
                card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
                if card.id == C.HARIYAMA: self.can_gust = True
            elif option.type == OptionType.RETREAT: self.can_switch = True
            elif option.type == OptionType.ATTACK:
                self.can_attack = True
                if option.attackId == MEGA_BRAVE: self.can_use_mega_brave = True

    def _my_board(self) -> list[Pokemon | None]: return self.me.active + self.me.bench
    def _opponent_board(self) -> list[Pokemon | None]: return self.opponent.active + self.opponent.bench
    def _opponent_has(self, ids: set[int]) -> bool: return any(pokemon is not None and pokemon.id in ids for pokemon in self._opponent_board())
    def _opponent_is_water_deck(self) -> bool: return self._opponent_has({C.KYOGRE, C.SNOVER, C.MEGA_ABOMASNOW_EX})
    def _opponent_is_crustle_wall(self) -> bool: return self._opponent_has({344, 345})

    def _can_evolve_board_index(self, board_index: int) -> bool:
        for option in self.select.option:
            if option.type != OptionType.EVOLVE: continue
            target_index = option.inPlayIndex + (1 if option.inPlayArea == AreaType.BENCH else 0)
            if target_index == board_index: return True
        return False

    def _base_attack(self, pokemon: Pokemon, attack_index: int) -> tuple[int, int, int] | None:
        energy_required, base_damage, base_score = 0, 0, 0
        if pokemon.id == C.MEGA_LUCARIO_EX:
            if attack_index == 0:
                energy_required, base_damage = 1, 130
                base_score += 60 * min(3, self.discard_counts[C.BASIC_FIGHTING_ENERGY])
            else:
                energy_required, base_damage = 2, 270
            if self._opponent_is_water_deck() and len(self.opponent.prize) <= 3: base_score -= 500
        elif attack_index == 1: return None
        elif pokemon.id == C.HARIYAMA: energy_required, base_damage = 3, 210
        elif pokemon.id == C.MAKUHITA: return None
        elif pokemon.id == C.SOLROCK and self.field_counts[C.LUNATONE] >= 1: energy_required, base_damage = 1, 70
        if base_damage <= 0: return None
        return energy_required, base_damage, base_score

    def _base_attack_after_evolution(self, pokemon: Pokemon, board_index: int, attack_index: int):
        if pokemon.id == C.MAKUHITA and attack_index == 0 and self._can_evolve_board_index(board_index): return 3, 210, -100
        return self._base_attack(pokemon, attack_index)

    def _plan_attack(self) -> None:
        global plan
        best_score = -1
        plan = AttackPlan()
        if self.state.turn < 2: return

        for attacker_index, my_pokemon in enumerate(self._my_board()):
            if my_pokemon is None: continue
            if attacker_index != 0 and not self.can_switch: break

            for attack_index in range(2):
                attack = self._base_attack_after_evolution(my_pokemon, attacker_index, attack_index)
                if attack is None: continue
                energy_required, base_damage, base_score = attack
                energy_count = len(my_pokemon.energies)
                if attack_index == 1 and attacker_index == 0 and energy_count >= 2 and not self.can_use_mega_brave: break
                needs_energy = False
                if energy_count < energy_required:
                    if self.hand_counts[C.BASIC_FIGHTING_ENERGY] >= 1 and not self.state.energyAttached:
                        energy_count += 1
                        needs_energy = energy_count >= energy_required
                    if not needs_energy: continue

                for target_index, op_pokemon in enumerate(self._opponent_board()):
                    if op_pokemon is None: continue
                    if target_index != 0 and not self.can_gust: break
                    if self._opponent_is_crustle_wall() and my_pokemon.id == C.MEGA_LUCARIO_EX and op_pokemon.id == 345: continue

                    damage = base_damage
                    op_data = card_table[op_pokemon.id]
                    if op_data.weakness == EnergyType.FIGHTING: damage *= 2
                    elif op_data.resistance == EnergyType.FIGHTING: damage -= 30

                    score = target_score(op_pokemon)
                    prize = prize_count(op_pokemon) if op_pokemon.hp <= damage else 0
                    if prize == 0: score *= damage / op_pokemon.hp
                    if len(self.opponent.prize) <= prize: score = 500000

                    score += base_score + (220 if attacker_index == 0 else 0) + (300 if target_index == 0 else 0) + energy_count
                    if score > best_score:
                        best_score = score
                        plan = AttackPlan(attacker_index, target_index, attack_index, op_pokemon.hp - damage, needs_energy)

    def _energy_target_score(self, pokemon: Pokemon, active: bool) -> int:
        energy_count = len(pokemon.energies)
        score = 8000 + (10 if active else 0)
        if pokemon.id in {C.MAKUHITA, C.HARIYAMA}:
            if pokemon.id == C.HARIYAMA: score += 1
            if self._opponent_is_crustle_wall(): score += 260 if energy_count < 3 else 30
            else: score += 100 if energy_count < 3 else 0; score -= 50 if self.has_ready_hariyama_line else 0
        elif pokemon.id == C.LUNATONE: score -= 100
        elif pokemon.id == C.SOLROCK: score += 20 if energy_count < 1 else -100
        elif pokemon.id in {C.RIOLU, C.MEGA_LUCARIO_EX}:
            if pokemon.id == C.MEGA_LUCARIO_EX: score += 1
            score += 100 if energy_count < 2 else 0
            score -= 50 if self.has_ready_lucario_line else 0
        return score

    def _score_option(self, option) -> float:
        if option.type == OptionType.NUMBER: return option.number
        if option.type == OptionType.YES: return 100 if self.context == SelectContext.IS_FIRST else 1
        if option.type == OptionType.NO: return 0
        if option.type == OptionType.CARD: return self._score_card_choice(option)
        if option.type == OptionType.PLAY: return self._score_play(option)
        if option.type == OptionType.ATTACH: return self._score_attach(option)
        if option.type == OptionType.EVOLVE: return self._score_evolve(option)
        if option.type == OptionType.ABILITY: return self._score_ability(option)
        if option.type == OptionType.RETREAT: return 2000 if plan.attacker >= 1 else -1
        if option.type == OptionType.ATTACK:
            return 1100 if (option.attackId == MEGA_BRAVE) == (plan.attack_index == 1) else 1000
        return 0

    def _score_card_choice(self, option) -> float:
        card = get_card(self.obs, option.area, option.index, option.playerIndex)
        if card is None: return 0
        if self.context in {SelectContext.SWITCH, SelectContext.TO_ACTIVE}: return self._score_active_choice(option, card)
        if self.context == SelectContext.SETUP_ACTIVE_POKEMON: return 2 if card.id == C.SOLROCK and self.state.firstPlayer == self.my_index else 4 if card.id == C.SOLROCK else 3 if card.id == C.RIOLU else 1 if card.id == C.MAKUHITA else 0
        if self.context == SelectContext.TO_HAND:
            score = 200 - self.hand_counts[card.id] * 100
            if card.id == C.MAKUHITA: score += (80 if self.field_counts[card.id] < 2 else -20) if self._opponent_is_crustle_wall() else (-10 if self.field_counts[card.id] >= 1 else 10)
            elif card.id == C.HARIYAMA: score += (120 if self.field_counts[C.MAKUHITA] >= 1 else -5) if self._opponent_is_crustle_wall() else (20 if self.field_counts[C.MAKUHITA] >= 1 else -20)
            elif card.id == C.LUNATONE: score += -250 if self.field_counts[card.id] >= 1 else 60
            elif card.id == C.SOLROCK: score += -250 if self.field_counts[card.id] >= 1 else 50
            elif card.id == C.RIOLU: score += -150 if (self.field_counts[C.RIOLU] + self.field_counts[C.MEGA_LUCARIO_EX] >= 2) else -3 if (self.field_counts[C.RIOLU] + self.field_counts[C.MEGA_LUCARIO_EX] >= 1) else 40
            elif card.id == C.MEGA_LUCARIO_EX: score += 40 if self.field_counts[C.RIOLU] >= 1 else -15
            elif card.id == C.BASIC_FIGHTING_ENERGY: score += 30 if not ability_used or not self.state.energyAttached else -1
            return score
        if self.context == SelectContext.ATTACH_FROM and isinstance(card, Pokemon): return self._energy_target_score(card, option.area == AreaType.ACTIVE)
        return 0

    def _score_active_choice(self, option, card: Pokemon | Card) -> float:
        if not isinstance(card, Pokemon): return 0
        if option.playerIndex != self.my_index: return 100 if option.index == plan.target - 1 else 0
        score = len(card.energies) * 2
        if option.index == plan.attacker - 1: score += 100
        if card.id == C.MEGA_LUCARIO_EX: score += 8 if self._opponent_is_water_deck() and len(self.opponent.prize) <= 3 else 20
        elif card.id == C.HARIYAMA and len(card.energies) >= 2: score += 45 if self._opponent_is_crustle_wall() else 15
        elif card.id == C.MAKUHITA and len(card.energies) >= 2: score += 35 if self._opponent_is_crustle_wall() else 10
        elif card.id == C.SOLROCK: score += 5
        elif card.id == C.RIOLU: score += 4
        return score

    def _score_play(self, option) -> float:
        card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
        data = card_table[card.id]
        if data.cardType == CardType.POKEMON:
            if card.id in {C.LUNATONE, C.SOLROCK} and self.field_counts[card.id] >= 1: return -1
            if card.id == C.RIOLU and self.field_counts[C.RIOLU] + self.field_counts[C.MEGA_LUCARIO_EX] >= 2: return -1
            return 20000
        if card.id == C.SWITCH: return 6000 if plan.attacker > 0 else -1
        if card.id == C.PREMIUM_POWER_PRO:
            if self.state.supporterPlayed and plan.remain_hp <= 0: return -1
            if not self.can_attack: return 3050 if (not self.state.supporterPlayed and self.hand_counts[C.CARMINE] > 0 and self.hand_counts[C.LILLIE_DETERMINATION] == 0 and not self.me.deckCount <= LOW_DECK_COUNT) else -1
            return 5000
        if card.id == C.BOSS_ORDERS: return 3200 if plan.target >= 1 else -1
        if card.id == C.CARMINE: 
            if self._opponent_is_crustle_wall() and any(c.id in {C.HARIYAMA, C.MAKUHITA} for c in self.me.hand): return -1
            return -1 if self.me.deckCount <= LOW_DECK_COUNT else 3000
        if card.id == C.LILLIE_DETERMINATION: 
            return -1 if self.me.deckCount <= LOW_DECK_COUNT else 3100
        if card.id == C.GRAVITY_MOUNTAIN: return 3500 if any(p is not None and card_table[p.id].stage2 for p in self._opponent_board()) else (1200 if self.stadium_id else -1)
        return 10000

    def _score_attach(self, option) -> float:
        card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
        pokemon = get_card(self.obs, option.inPlayArea, option.inPlayIndex, self.my_index)
        if not isinstance(pokemon, Pokemon): return 0
        if card.id == C.HERO_CAPE:
            score = 7000
            if self._opponent_is_water_deck(): return 12200 if pokemon.id == C.RIOLU else 12800 if pokemon.id == C.MEGA_LUCARIO_EX else score
            if pokemon.id == C.RIOLU: score += 100
            elif pokemon.id == C.MEGA_LUCARIO_EX: score += 200
            return score
        score = self._energy_target_score(pokemon, option.inPlayArea == AreaType.ACTIVE)
        board_index = option.inPlayIndex if option.inPlayArea == AreaType.ACTIVE else option.inPlayIndex + 1
        if board_index == plan.attacker and plan.needs_energy: score += 200
        return score

    def _score_evolve(self, option) -> float:
        pokemon = get_card(self.obs, option.inPlayArea, option.inPlayIndex, self.my_index)
        if not isinstance(pokemon, Pokemon): return 0
        if pokemon.id == C.MAKUHITA and plan.target == 0 and not self._opponent_is_crustle_wall(): return -1
        return 9000 + len(pokemon.energies)

    def _score_ability(self, option) -> float:
        card = get_card(self.obs, option.area, option.index, self.my_index)
        if card.id == C.LUMIOSE_CITY: return 1
        if card.id == C.LUNATONE and self.me.deckCount <= LOW_DECK_COUNT: return -1
        return 30000

    def _remember_lunatone_ability(self, ranked: list[int]) -> None:
        global ability_used
        if self.context != SelectContext.MAIN or not ranked: return
        option = self.select.option[ranked[0]]
        if option.type != OptionType.ABILITY: return
        card = get_card(self.obs, option.area, option.index, self.my_index)
        if card is not None and card.id == C.LUNATONE: ability_used = True

def evaluate_state(obs):
    st = obs.current
    if st is None: return 0.0
    me, op = st.players[st.yourIndex], st.players[1 - st.yourIndex]
    
    prize_diff = len(op.prize) - len(me.prize)
    if len(me.prize) == 0: return 9999999.0
    if len(op.prize) == 0: return -9999999.0
    val = prize_diff * 10000.0
    
    is_crustle = any(p is not None and p.id in {344, 345} for p in [op.active[0] if op.active else None] + list(op.bench))
    is_snorlax = any(p is not None and p.id == 143 for p in [op.active[0] if op.active else None] + list(op.bench))
    is_stall = is_crustle or is_snorlax
    
    # 1. My Board Strength
    for p in [me.active[0] if me.active else None] + list(me.bench):
        if p is None: continue
        val += len(p.energies) * 200.0
        if is_crustle:
            if p.id == C.HARIYAMA: val += 1500.0
            elif p.id == C.MAKUHITA: val += 800.0
            elif p.id == C.MEGA_LUCARIO_EX: val += 0.0
            elif p.id == C.RIOLU: val += 0.0
        else:
            if p.id == C.MEGA_LUCARIO_EX: val += 500.0
            elif p.id == C.HARIYAMA: val += 300.0
            elif p.id in {C.RIOLU, C.MAKUHITA}: val += 100.0
            
    # 1.5. Hand Conservation (Crucial against Stall)
    if is_crustle:
        for c in me.hand:
            if c.id == C.HARIYAMA: val += 1000.0
            elif c.id == C.MAKUHITA: val += 500.0
        
    if me.active and me.active[0] is not None: 
        val += me.active[0].hp * 2.0
        if len(me.active[0].energies) >= 2: val += 500.0
        
    # 2. Predictive Threat Mapping (Assume opponent attaches 1 energy next turn)
    op_max_damage = 0
    for p in [op.active[0] if op.active else None] + list(op.bench):
        if p is None: continue
        val -= p.hp * 1.5
        
        assumed_energies = len(p.energies) + 1
        op_dmg = 0
        if p.id == C.MEGA_LUCARIO_EX: op_dmg = 270 if assumed_energies >= 2 else 130 if assumed_energies >= 1 else 0
        elif p.id == C.HARIYAMA: op_dmg = 210 if assumed_energies >= 3 else 0
        elif p.id == C.KYOGRE: op_dmg = 130 if assumed_energies >= 3 else 0
        elif p.id == C.MEGA_ABOMASNOW_EX: op_dmg = 240 if assumed_energies >= 4 else 0
        elif p.id == 121: op_dmg = 130 if assumed_energies >= 2 else 0 # Dragapult ex
        else: op_dmg = assumed_energies * 40
        op_max_damage = max(op_max_damage, op_dmg)
        
    # 3. Lethal Threat Penalty
    if me.active and me.active[0] is not None:
        my_active = me.active[0]
        if op_max_damage >= my_active.hp:
            prize_risk = 2 if my_active.id == C.MEGA_LUCARIO_EX else 1
            val -= prize_risk * 4000.0
        elif op_max_damage > 0:
            val -= op_max_damage * 1.5

    # 4. Anti-Stall Deck Conservation
    deck_c = getattr(me, "deckCount", 60)
    if is_stall:
        val += deck_c * 30.0
        val += getattr(me, "handCount", len(me.hand)) * 2.0
    else:
        val += getattr(me, "handCount", len(me.hand)) * 10.0
        
    if deck_c < 5: val -= 10000.0
    return val

def rollout_turn(sid, cur_obs, your_index):
    steps = 0
    while steps < 20:
        if cur_obs.current.result is not None and cur_obs.current.result != -1: break
        if cur_obs.current.yourIndex != your_index: break
        if cur_obs.select.context != SelectContext.MAIN:
            sub = AdvancedPolicy(cur_obs).choose()
            sel = sub[: max(1, cur_obs.select.minCount)]
        else:
            nxt = AdvancedPolicy(cur_obs).choose()
            if not nxt: break
            sel = [nxt[0]]
            if cur_obs.select.option[nxt[0]].type == OptionType.END:
                search_step(sid, sel)
                break
        ar = search_step(sid, sel)
        if getattr(ar, "error", 0) != 0 or ar.state is None: break
        cur_obs, sid = ar.state.observation, ar.state.searchId
        steps += 1
    return cur_obs


def simulate_action(obs, action):
    my_p = obs.current.players[obs.current.yourIndex]
    yd = random.sample(my_deck, getattr(my_p, "deckCount", 60))
    sbi = search_begin(obs, your_deck=yd)
    if getattr(sbi, "error", 0) != 0 or sbi.state is None: return -float('inf')
    ar = search_step(sbi.state.searchId, [action])
    if getattr(ar, "error", 0) != 0 or ar.state is None: return -float('inf')
    cur = rollout_turn(ar.state.searchId, ar.state.observation, obs.current.yourIndex)
    return evaluate_state(cur)

def SEARCH_ALGO(obs_dict, obs):
    if not (_SEARCH_OK and USE_SEARCH): return None
    select = obs.select
    if select is None or select.context != SelectContext.MAIN: return None
    t0 = time.time()
    
    base_order = AdvancedPolicy(obs).choose()
    candidates = base_order[:SEARCH_MAX_CANDIDATES]
    if not candidates: return None
    if len(candidates) == 1: return [candidates[0]] + [i for i in base_order if i != candidates[0]]
    
    import math
    visits = {a: 0 for a in candidates}
    total_val = {a: 0.0 for a in candidates}
    
    try:
        # Phase 1: Exploration
        for a in candidates:
            if time.time() - t0 > SEARCH_TIME_BUDGET: break
            val = simulate_action(obs, a)
            if val != -float('inf'):
                visits[a] += 1
                total_val[a] += val
                
        # Phase 2: UCB1
        while time.time() - t0 < SEARCH_TIME_BUDGET:
            total_visits = sum(visits.values())
            if total_visits == 0: break
            
            valid_scores = [total_val[a]/visits[a] for a in candidates if visits[a] > 0]
            if not valid_scores: break
            min_s = min(valid_scores)
            max_s = max(valid_scores)
            if max_s == min_s: max_s = min_s + 1.0
            
            best_ucb, best_a = -float('inf'), candidates[0]
            for a in candidates:
                if visits[a] == 0:
                    best_a = a
                    break
                avg = total_val[a] / visits[a]
                norm_avg = (avg - min_s) / (max_s - min_s)
                ucb = norm_avg + 0.5 * math.sqrt(math.log(total_visits) / visits[a])
                if ucb > best_ucb:
                    best_ucb = ucb
                    best_a = a
                    
            val = simulate_action(obs, best_a)
            if val != -float('inf'):
                visits[best_a] += 1
                total_val[best_a] += val
                
        best_action = max(candidates, key=lambda a: total_val[a]/visits[a] if visits[a] > 0 else -float('inf'))
        return [best_action] + [i for i in base_order if i != best_action]
    except Exception: return None


def agent(obs_dict: dict) -> list[int]:
    try: obs = to_observation_class(obs_dict)
    except Exception: return my_deck if obs_dict.get("select") is None else [0]
    if obs.select is None: return my_deck
    
    global pre_turn, ability_used, plan
    if pre_turn != obs.current.turn:
        pre_turn = obs.current.turn
        ability_used = False
        plan = AttackPlan()

    try:
        ordered = SEARCH_ALGO(obs_dict, obs)
        if ordered is None: ordered = AdvancedPolicy(obs).choose()
        n = len(obs.select.option)
        ordered = [i for i in ordered if 0 <= i < n]
        if not ordered: return list(range(min(max(1, obs.select.minCount), n)))
        k = max(min(obs.select.maxCount, n), min(max(1, obs.select.minCount), n))
        return ordered[:k]
    except Exception:
        n = len(obs.select.option)
        return list(range(min(max(1, obs.select.minCount), n)))








In [ ]:
%%writefile orig_archaludon.py
"""Archaludon ex + Cinderace — Rule-based agent (Public version)

Deck Concept:
  Cinderace's Explosiveness places it face-down as Active during setup.
  Turn 1 Turbo Flare ({C}=50) accelerates up to 3 Basic Energy from deck
  to benched Duraludon. Evolving into Archaludon ex triggers Assemble Alloy,
  attaching up to 2 Basic Metal Energy from discard to Metal Pokemon.
  Metal Defender ({M}{M}{M}=220) is the main attack; no Weakness next turn.
  Duraludon can attack directly with Raging Hammer ({M}{M}{C}=80 + 10 per
  damage counter) without evolving. Relicanth's Memory Dive also unlocks
  Raging Hammer on Archaludon ex after evolution. Hero's Cape gives +100 HP
  (HP400). Full Metal Lab reduces attack damage to Metal Pokemon by 30.

Pokemon:
  Duraludon (169)      - Basic Metal HP130. Hammer In {M}=30.
                         Raging Hammer {M}{M}{C}=80+10*damage_counters.
  Archaludon ex (190)  - Stage 1 from Duraludon, HP300. Assemble Alloy: on evolve
                         from hand, attach up to 2 Metal Energy from discard.
                         Metal Defender {M}{M}{M}=220, no Weakness next turn.
  Cinderace (666)      - Stage 2 HP160. Explosiveness: place face-down as Active
                         in setup from opening hand. Turbo Flare {C}=50, attach
                         up to 3 Basic Energy from deck to benched Pokemon.
  Relicanth (57)       - Basic HP100. Memory Dive: evolved Pokemon can use attacks
                         from previous Evolutions. Archaludon ex -> Raging Hammer.

Trainers:
  Poke Pad (1152), Ultra Ball (1121), Pokegear 3.0 (1122), Night Stretcher (1097),
  Jumbo Ice Cream (1147), Hero's Cape (1159), Boss's Orders (1182),
  Explorer's Guidance (1185), Lillie's Determination (1227), Full Metal Lab (1244) x4.

Energy: Basic Metal Energy (8) x11

Score system:
  Setup/play/evolve/attach: 1000~28000 (high = do first)
  Attack: damage value (always last — attacking ends the turn)
  Negative = skip if above minCount
"""

import os
import random
import sys

try:
    ROOT = __file__
except NameError:
    ROOT = None
CG_PATH = "/kaggle_simulations/agent"
for p in ([os.path.dirname(os.path.abspath(ROOT))] if ROOT else []) + [CG_PATH]:
    if p and p not in sys.path and os.path.isdir(p):
        sys.path.insert(0, p)

from cg.api import (
    AreaType,
    LogType,
    OptionType,
    SelectContext,
    all_card_data,
    to_observation_class,
)

try:
    from cg.api import all_attack
    ALL_ATTACKS = {a.attackId: a for a in all_attack()}
except Exception:
    ALL_ATTACKS = {}

# ── Card IDs ──

DURALUDON = 169
ARCHALUDON_EX = 190
CINDERACE = 666
RELICANTH = 57
CRUSTLE_LINE = {344, 345, 532}
STARMIE_LINE = {1030, 1031}
LUCARIO_LINE = {677, 678}
HOP_LINE = {288, 289, 299, 304, 307, 308, 309, 310, 878, 879}
HOP_SNORLAX = 304

METAL_ENERGY = 8

POKE_PAD = 1152
ULTRA_BALL = 1121
POKEGEAR = 1122
NIGHT_STRETCHER = 1097
JUMBO_ICE_CREAM = 1147
HERO_CAPE = 1159
BOSS = 1182
EXPLORER = 1185
LILLIE = 1227
FULL_METAL_LAB = 1244

RAGING_HAMMER = 224
METAL_DEFENDER = 253

_ATTACK_BASE_DMG = {METAL_DEFENDER: 220, 965: 50, 223: 30, 61: 30}

_SETUP_ACTIVE_PRIORITY = {
    CINDERACE: (100000, "Active: Cinderace Explosiveness"),
    DURALUDON: (20000, "Active fallback: Duraludon"),
    RELICANTH: (5000, "Active fallback: Relicanth"),
}

ALWAYS_SAFE_DISCARD = {METAL_ENERGY, CINDERACE}

CARD_DB = {c.cardId: c for c in all_card_data()}

MEGA_BRAVE = 983
PREMIUM_POWER_PRO = 1141
HARIYAMA_LINE = {673, 674}

# Track opponent's last-turn attack via logs
_opp_last_attack_id = None
_cur_turn_logs = []


def _update_opp_attack_tracking(obs):
    global _opp_last_attack_id, _cur_turn_logs
    yi = obs.current.yourIndex
    for entry in obs.logs:
        if entry.type == LogType.TURN_END:
            for prev in _cur_turn_logs:
                if prev.type == LogType.ATTACK and getattr(prev, 'playerIndex', yi) != yi:
                    _opp_last_attack_id = prev.attackId
            _cur_turn_logs.clear()
        else:
            _cur_turn_logs.append(entry)


# ── Board helpers ──

def read_deck_csv():
    fp = "deck.csv"
    if not os.path.exists(fp):
        fp = "/kaggle_simulations/agent/deck.csv"
    with open(fp) as f:
        return [int(line) for line in f.read().strip().split("\n")]


def get_card(obs, area, index, player_index):
    if area is None or index is None:
        return None
    ps = obs.current.players[player_index]
    if area == AreaType.DECK and obs.select and obs.select.deck is not None:
        return obs.select.deck[index] if index < len(obs.select.deck) else None
    if area == AreaType.HAND and ps.hand is not None:
        return ps.hand[index] if index < len(ps.hand) else None
    if area == AreaType.DISCARD:
        return ps.discard[index] if index < len(ps.discard) else None
    if area == AreaType.ACTIVE:
        return ps.active[index] if index < len(ps.active) else None
    if area == AreaType.BENCH:
        return ps.bench[index] if index < len(ps.bench) else None
    if area == AreaType.PRIZE:
        return ps.prize[index] if index < len(ps.prize) else None
    if area == AreaType.STADIUM:
        return obs.current.stadium[index] if index < len(obs.current.stadium) else None
    if area == AreaType.LOOKING and obs.current.looking is not None:
        return obs.current.looking[index] if index < len(obs.current.looking) else None
    return None


def option_card(obs, opt):
    yi = obs.current.yourIndex
    pi = opt.playerIndex if opt.playerIndex is not None else yi
    if opt.type == OptionType.PLAY:
        return get_card(obs, AreaType.HAND, opt.index, pi)
    return get_card(obs, opt.area, opt.index, pi)


def option_target(obs, opt):
    if opt.inPlayArea is None or opt.inPlayIndex is None:
        return None
    return get_card(obs, opt.inPlayArea, opt.inPlayIndex, obs.current.yourIndex)


def my_state(obs):
    return obs.current.players[obs.current.yourIndex]


def opp_state(obs):
    return obs.current.players[1 - obs.current.yourIndex]


def active_pokemon(obs):
    ps = my_state(obs)
    return ps.active[0] if ps.active else None


def opp_active_pokemon(obs):
    ps = opp_state(obs)
    return ps.active[0] if ps.active else None


def opp_bench_pokemon(obs):
    return [p for p in opp_state(obs).bench if p]


def all_my_pokemon(obs):
    ps = my_state(obs)
    return [p for p in (ps.active + ps.bench) if p]


def hand_ids(obs):
    hand = my_state(obs).hand
    return [c.id for c in hand if c] if hand else []


def discard_ids(obs):
    return [c.id for c in (my_state(obs).discard or []) if c]


def metal_in_discard(obs):
    return sum(1 for c in (my_state(obs).discard or []) if c and c.id == METAL_ENERGY)


def energy_count(pokemon):
    if pokemon is None:
        return 0
    if getattr(pokemon, "energyCards", None) is not None:
        return len(pokemon.energyCards)
    return len(getattr(pokemon, "energies", []) or [])


def retreat_cost(pokemon):
    data = CARD_DB.get(pokemon.id) if pokemon else None
    return getattr(data, "retreatCost", 0) if data else 0


def damage_on(pokemon):
    if pokemon is None:
        return 0
    return max(0, getattr(pokemon, "maxHp", pokemon.hp) - pokemon.hp)


def has_tool(pokemon):
    return bool(getattr(pokemon, "tools", []) or [])


def count_in_play(obs, card_id):
    return sum(1 for p in all_my_pokemon(obs) if p.id == card_id)


def has_in_play(obs, card_id):
    return any(p.id == card_id for p in all_my_pokemon(obs))


def need_duraludon(obs):
    return sum(1 for p in all_my_pokemon(obs) if p.id in {DURALUDON, ARCHALUDON_EX}) < 2


def need_archaludon(obs):
    has_dura, ex_count = False, 0
    for p in all_my_pokemon(obs):
        if p.id == DURALUDON:
            has_dura = True
        elif p.id == ARCHALUDON_EX:
            ex_count += 1
    return has_dura and ex_count < 2


def safe_discard_count(obs):
    ids = hand_ids(obs)
    mt = metal_in_discard(obs)
    safe = 0
    for cid in ids:
        if cid == METAL_ENERGY and mt + safe < 2:
            safe += 1
        elif cid == CINDERACE:
            safe += 1
    draw_in_hand = sum(1 for c in ids if c in (LILLIE, EXPLORER))
    if draw_in_hand >= 2:
        safe += draw_in_hand - 1
    return safe


def prize_value(pokemon):
    data = CARD_DB.get(pokemon.id) if pokemon else None
    if data and getattr(data, "megaEx", False):
        return 3
    if data and getattr(data, "ex", False):
        return 2
    return 1


def best_attack_damage(obs, attack_id):
    if attack_id == RAGING_HAMMER:
        return 80 + damage_on(active_pokemon(obs)) // 10 * 10
    return _ATTACK_BASE_DMG.get(attack_id, 0)


def is_metal_weak(pokemon):
    if pokemon is None:
        return False
    data = CARD_DB.get(pokemon.id)
    w = getattr(data, "weakness", None) if data else None
    if w is None:
        return False
    return getattr(w, "value", w) == METAL_ENERGY


def effective_damage(base_damage, target):
    return base_damage * 2 if is_metal_weak(target) else base_damage


def _first_option_index(obs, card_id):
    for o in obs.select.option:
        oc = option_card(obs, o)
        if oc and oc.id == card_id:
            return getattr(o, 'index', None)
    return None


# ── Attack routes ──

def direct_attack_energy_route(obs, pokemon):
    e = energy_count(pokemon)
    if e >= 3:
        return True, False
    if e == 2 and not obs.current.energyAttached and METAL_ENERGY in hand_ids(obs):
        return True, True
    return False, False


def can_evolve_to_archaludon_now(pokemon, obs):
    if pokemon is None or pokemon.id != DURALUDON:
        return False
    if ARCHALUDON_EX not in hand_ids(obs):
        return False
    return not getattr(pokemon, "appearThisTurn", True)


def alloy_attack_energy_route(obs, pokemon):
    if not can_evolve_to_archaludon_now(pokemon, obs):
        return False, False
    current = energy_count(pokemon)
    alloy = min(2, metal_in_discard(obs))
    total = current + alloy
    if total >= 3:
        return True, False
    if total == 2 and not obs.current.energyAttached and METAL_ENERGY in hand_ids(obs):
        return True, True
    return False, False


def attack_energy_route(obs, pokemon):
    if pokemon is None:
        return False, False
    if pokemon.id == ARCHALUDON_EX:
        return direct_attack_energy_route(obs, pokemon)
    if pokemon.id == DURALUDON:
        ok, uses_attach = direct_attack_energy_route(obs, pokemon)
        if ok:
            return True, uses_attach
        return alloy_attack_energy_route(obs, pokemon)
    return False, False


def archaludon_ex_attack_route(obs):
    active = active_pokemon(obs)
    if active and active.id in {ARCHALUDON_EX, DURALUDON}:
        ok, uses_attach = attack_energy_route(obs, active)
        if ok:
            return {"attacker": active, "uses_attach": uses_attach, "needs_retreat": False}

    if active is None or obs.current.retreated or energy_count(active) < retreat_cost(active):
        return None
    ps = my_state(obs)
    for pokemon in [p for p in ps.bench if p]:
        if pokemon.id not in {ARCHALUDON_EX, DURALUDON}:
            continue
        ok, uses_attach = attack_energy_route(obs, pokemon)
        if ok:
            return {"attacker": pokemon, "uses_attach": uses_attach, "needs_retreat": True}
    return None


def planned_archaludon_attacks(obs):
    route = archaludon_ex_attack_route(obs)
    if route is None:
        return []
    attacker = route["attacker"]
    attacks = []
    if attacker.id == ARCHALUDON_EX:
        attacks.append({"damage": 220})
        if has_in_play(obs, RELICANTH):
            attacks.append({"damage": 80 + damage_on(attacker) // 10 * 10})
    if attacker.id == DURALUDON:
        attacks.append({"damage": 80 + damage_on(attacker) // 10 * 10})
        if can_evolve_to_archaludon_now(attacker, obs):
            attacks.append({"damage": 220})
    return attacks


# ── Matchup detection & opponent max damage ──

ALAKAZAM_LINE = {741, 742, 743}
_ALA_BOARD_GAIN = {66: 3, 742: 2, 305: 2, 65: 2, 741: 1}  # Dudunsparce, Kadabra, Dunsparce×2, Abra


def _estimate_alakazam_from_pokes(opp, pokes):
    """(floor, ceiling, ceiling_with_boss) damage from visible Alakazam line."""
    ids = [p.id for p in pokes if p]
    if not (ALAKAZAM_LINE & set(ids)):
        return 0, 0, 0
    base = opp.handCount + 1
    gain = sum(_ALA_BOARD_GAIN.get(i, 0) for i in ids)
    enriching_seen = (
        any(c and c.id == 13 for c in (opp.discard or []))
        or any(c and c.id == 13 for p in pokes if p for c in (getattr(p, "energyCards", None) or []))
    )
    if not enriching_seen:
        gain += 3
    if any(i == 140 for i in ids):
        gain += 3
    return base * 20, (base + gain + 2) * 20, (base + gain - 1) * 20


def _estimate_alakazam(obs):
    """(floor, ceiling, ceiling_with_boss) damage from Powerful Hand."""
    opp = opp_state(obs)
    pokes = ([opp.active[0]] if opp.active else []) + list(opp.bench or [])
    return _estimate_alakazam_from_pokes(opp, pokes)


def detect_matchup(obs):
    opp = opp_state(obs)
    ids = {p.id for p in (opp.active + opp.bench) if p}
    if ids & CRUSTLE_LINE:
        return "crustle"
    if ids & HOP_LINE:
        return "hop"
    if ids & STARMIE_LINE:
        return "starmie"
    if ids & LUCARIO_LINE:
        return "lucario"
    if ids & ALAKAZAM_LINE:
        return "alakazam"
    return "generic"


def opp_max_damage(obs):
    matchup = detect_matchup(obs)
    if matchup == "alakazam":
        _, ceiling, _ = _estimate_alakazam(obs)
        return ceiling
    if matchup == "crustle":
        return 120
    if matchup == "hop":
        return 220
    if matchup == "lucario":
        return 270  # Mega Brave base. PPP adds +30 each but unpredictable
    if matchup == "starmie":
        return 210
    return 220


# ── Overrides ──

def apply_overrides(obs, opt, score, reason):
    # Hard rule: don't Explorer with low deck
    if opt.type == OptionType.PLAY:
        card = option_card(obs, opt)
        cid = card.id if card else None
        if my_state(obs).deckCount <= 10 and cid == EXPLORER:
            return -5000, "hard: don't Explorer with low deck"

    if detect_matchup(obs) != "crustle":
        return score, reason

    # Crustle overrides
    card = option_card(obs, opt)
    cid = card.id if card else getattr(opt, 'cardId', None)
    ctx = obs.select.context

    if opt.type == OptionType.EVOLVE and cid == ARCHALUDON_EX:
        return -10000, "Crustle: don't evolve to ex"

    if opt.type == OptionType.ATTACK:
        aid = getattr(opt, 'attackId', None)
        active = active_pokemon(obs)
        opp_act = opp_active_pokemon(obs)
        opp_has_spiky = bool(opp_act and any(
            getattr(c, 'id', None) == 14
            for c in (getattr(opp_act, 'energyCards', None) or [])))
        if (active and active.id == DURALUDON and active.hp == 130
                and opp_act and opp_act.id == 345 and energy_count(opp_act) >= 2
                and opp_has_spiky):
            return -3000, "Crustle: full HP Duraludon waits out Spiky"
        if aid == METAL_DEFENDER:
            return -5000, "Crustle: Metal Defender does 0"
        if aid == RAGING_HAMMER:
            rh_dmg = 80 + damage_on(active_pokemon(obs)) // 10 * 10
            return max(score, 200), "Crustle: Raging Hammer"

    if opt.type == OptionType.PLAY:
        if cid == RELICANTH:
            return -5000, "Crustle: skip Relicanth"
        dc = my_state(obs).deckCount
        if dc <= 10 and cid in (EXPLORER, LILLIE):
            if cid == LILLIE and dc <= 3 and my_state(obs).handCount >= dc + 6:
                return 15000, "Crustle: Lillie to refill deck"
            return -5000, "Crustle: don't draw with low deck"
        if cid == LILLIE:
            has_metal = any(c and c.id == METAL_ENERGY for c in (my_state(obs).hand or []) if c)
            if not has_metal:
                return score, "Crustle: Lillie OK (no energy in hand)"

    if opt.type == OptionType.ATTACH:
        target = option_target(obs, opt)
        tid = target.id if target else None
        if getattr(opt, 'inPlayArea', None) == AreaType.BENCH and tid == DURALUDON:
            return score + 10000, "Crustle: bench Duraludon energy priority"
        if getattr(opt, 'inPlayArea', None) == AreaType.ACTIVE:
            active = active_pokemon(obs)
            if active and energy_count(active) >= 2:
                return score + 3000, "Crustle: Active 3rd energy"

    if ctx == SelectContext.TO_HAND and opt.type == OptionType.CARD and cid == ARCHALUDON_EX:
        return -3000, "Crustle: skip Archaludon ex"

    if ctx in {SelectContext.DISCARD, SelectContext.DISCARD_CARD_OR_ATTACHED_CARD}:
        if cid == ARCHALUDON_EX and score < 0:
            return 9000, "Crustle: discard Archaludon ex"

    return score, reason


# ── Scoring ──

def score_setup(obs, opt):
    card = option_card(obs, opt)
    cid = card.id if card else None
    ctx = obs.select.context

    if ctx == SelectContext.MULLIGAN:
        return (10000, "no mulligan") if opt.type == OptionType.NO else (0, "mulligan")
    if ctx == SelectContext.IS_FIRST:
        return (10000, "choose second") if opt.type == OptionType.NO else (0, "go first")
    if ctx == SelectContext.SETUP_ACTIVE_POKEMON:
        return _SETUP_ACTIVE_PRIORITY.get(cid, (0, "unknown Active"))
    if ctx == SelectContext.SETUP_BENCH_POKEMON:
        return -10000, "never bench during setup"
    return 0, "non-setup"


# HP threshold per matchup: skip Ice Cream if HP > this value
_ICE_CREAM_HP_THRESHOLD = {
    "lucario": 270,
    "starmie": 210,
    "crustle": 120,
    "hop": 220,
    "generic": 230,
}


def should_skip_ice_cream(obs, active):
    """Decide whether to skip Jumbo Ice Cream. Returns (skip: bool, reason: str)."""
    # 1. Active must be Archaludon ex
    if active.id != ARCHALUDON_EX:
        return True, "skip Ice Cream: not Archaludon ex"
    # 2. Raging Hammer KO guard: don't heal if it loses a KO (but 220 Metal Defender still KOs → heal OK)
    opp_act = opp_active_pokemon(obs)
    if opp_act and has_in_play(obs, RELICANTH):
        md_kills = effective_damage(220, opp_act) >= opp_act.hp
        if not md_kills:
            rh_dmg = 80 + damage_on(active) // 10 * 10
            rh_after = 80 + max(0, damage_on(active) - 80) // 10 * 10
            if effective_damage(rh_dmg, opp_act) >= opp_act.hp and effective_damage(rh_after, opp_act) < opp_act.hp:
                return True, "skip Ice Cream: healing loses Raging Hammer KO"
    # 3. Alakazam: all-or-nothing Ice Cream decision
    matchup = detect_matchup(obs)
    if matchup == "alakazam":
        floor, ceiling, _ = _estimate_alakazam(obs)
        opp_a = opp_active_pokemon(obs)
        attacks = planned_archaludon_attacks(obs)
        if opp_a and attacks and any(effective_damage(a["damage"], opp_a) >= opp_a.hp for a in attacks):
            _, ceiling, _ = _estimate_alakazam_from_pokes(opp_state(obs), opp_bench_pokemon(obs))
        ice_count = sum(1 for c in (my_state(obs).hand or []) if c and c.id == JUMBO_ICE_CREAM)
        max_hp = getattr(active, "maxHp", active.hp)
        hp_after_all = min(max_hp, active.hp + ice_count * 80)
        if hp_after_all <= active.hp:
            return True, "skip Ice Cream: no effective healing"
        if hp_after_all < floor:
            return True, f"skip Ice Cream: even {ice_count}x heal ({hp_after_all}) < floor {floor}"
        if hp_after_all >= ceiling:
            return False, f"use Ice Cream: {ice_count}x heal ({hp_after_all}) >= ceil {ceiling}"
        return False, f"use Ice Cream: {ice_count}x heal ({hp_after_all}) between floor={floor} ceil={ceiling}"
    # 4. HP above matchup threshold
    threshold = _ICE_CREAM_HP_THRESHOLD.get(matchup, 220)
    if active.hp > threshold:
        return True, f"skip Ice Cream: HP {active.hp} > {threshold} ({matchup})"
    # 5. Use it
    return False, ""


ITEMS = {POKE_PAD, ULTRA_BALL, POKEGEAR, NIGHT_STRETCHER, JUMBO_ICE_CREAM, HERO_CAPE}


def score_play(obs, opt):
    card = option_card(obs, opt)
    cid = card.id if card else None
    ids = hand_ids(obs)

    # ── Pokemon: bench if available ──
    if cid in {DURALUDON, RELICANTH}:
        return 18000, "play Pokemon"

    # ── Stadium ──
    if cid == FULL_METAL_LAB:
        active = active_pokemon(obs)
        if active and active.id not in {DURALUDON, ARCHALUDON_EX}:
            return -200, "skip FML: Active not Metal"
        return 20000, "play Full Metal Lab"

    # ── Items: default 20000, only negative exceptions ──
    if cid in ITEMS:
        if cid == HERO_CAPE:
            if not any(p.id in {ARCHALUDON_EX, DURALUDON} and not has_tool(p) for p in all_my_pokemon(obs)):
                return -500, "save Hero's Cape: no target"
        if cid == JUMBO_ICE_CREAM:
            active = active_pokemon(obs)
            if active:
                skip, reason = should_skip_ice_cream(obs, active)
                if skip:
                    return -500, reason
        if cid == NIGHT_STRETCHER:
            disc = discard_ids(obs)
            has_urgent = (
                (DURALUDON in disc and DURALUDON not in ids and count_in_play(obs, DURALUDON) + count_in_play(obs, ARCHALUDON_EX) <= 1)
                or (ARCHALUDON_EX in disc and ARCHALUDON_EX not in ids and has_in_play(obs, DURALUDON))
                or (METAL_ENERGY in disc and not obs.current.energyAttached
                    and sum(1 for c in (my_state(obs).hand or []) if c and c.id == METAL_ENERGY) == 0
                    and any(p and p.id in (DURALUDON, ARCHALUDON_EX) and energy_count(p) == 2 for p in all_my_pokemon(obs)))
            )
            if not has_urgent:
                return -500, "save Night Stretcher"
        if cid == ULTRA_BALL:
            bench_empty = len([p for p in my_state(obs).bench if p]) == 0
            if bench_empty:
                return 300, "Ultra Ball: bench empty (donk risk)"
            metal_in_hand = sum(1 for c in (my_state(obs).hand or []) if c and c.id == METAL_ENERGY)
            metal_in_trash = metal_in_discard(obs)
            if metal_in_trash == 0 and metal_in_hand >= 1:
                return 20000, "Ultra Ball: fuel Alloy"
            if safe_discard_count(obs) >= 2 and (need_archaludon(obs) or need_duraludon(obs)):
                return 20000, "Ultra Ball: search line"
            return -1000, "skip Ultra Ball"
        return 20000, "play item"

    if cid == EXPLORER:
        if obs.current.supporterPlayed:
            return -1000, "Supporter already used"
        return 16000, "play Explorer"

    if cid == LILLIE:
        if obs.current.supporterPlayed:
            return -1000, "Supporter already used"
        if BOSS in ids and planned_archaludon_attacks(obs):
            return -500, "save Lillie: Boss in hand with attacker ready"
        return 5000, "play Lillie"

    if cid == BOSS:
        if obs.current.supporterPlayed:
            return -1000, "Supporter already used"
        # vs Hop: Boss Snorlax to remove Extra Helpings (+30) ASAP
        if detect_matchup(obs) == "hop":
            active = active_pokemon(obs)
            opp_has_snorlax = any(p.id == HOP_SNORLAX for p in opp_bench_pokemon(obs))
            if opp_has_snorlax and active:
                # Case 1: Cinderace active + bench has Duraludon → Turbo Flare Snorlax
                if active.id == CINDERACE:
                    has_dura_bench = any(p.id in {DURALUDON, ARCHALUDON_EX}
                                        for p in my_state(obs).bench if p)
                    if has_dura_bench:
                        return 16500, "Boss: pull Snorlax (Cinderace Turbo Flare)"
                # Case 2: Archaludon active, HP > 220, can attack → Boss Snorlax
                if active.id == ARCHALUDON_EX and active.hp > 220:
                    ok, _ = attack_energy_route(obs, active)
                    if ok:
                        return 16500, "Boss: pull Snorlax (Arch can tank Revenge 220)"
        if _opp_last_attack_id == MEGA_BRAVE:
            return -500, "save Boss: Mega Brave stuck"
        attacks = planned_archaludon_attacks(obs)
        if not attacks:
            return -500, "save Boss: no attacker"
        opp_act = opp_active_pokemon(obs)
        can_ko_active = opp_act and any(
            effective_damage(atk["damage"], opp_act) >= opp_act.hp for atk in attacks)
        remaining = len(my_state(obs).prize)
        if can_ko_active:
            if prize_value(opp_act) >= remaining:
                return -500, "save Boss: Active KO wins"
            for target in opp_bench_pokemon(obs):
                for atk in attacks:
                    if effective_damage(atk["damage"], target) >= target.hp:
                        if prize_value(target) >= remaining:
                            return 20000, "LETHAL Boss"
                        break
            return -500, "save Boss: can KO Active"
        best_score = -500
        best_reason = "save Boss"
        for target in opp_bench_pokemon(obs):
            for atk in attacks:
                if effective_damage(atk["damage"], target) >= target.hp:
                    pv = prize_value(target)
                    if pv >= remaining:
                        return 20000, "LETHAL Boss"
                    s = 4000 + pv * 200 + energy_count(target) * 100
                    if s > best_score:
                        best_score = s
                        best_reason = "Boss: pull bench target"
                    break
        if best_score <= 0:
            metal_total = sum(1 for c in (my_state(obs).hand or []) if c and c.id == METAL_ENERGY)
            metal_total += sum(energy_count(p) for p in all_my_pokemon(obs) if p)
            has_cind = has_in_play(obs, CINDERACE)
            draw_in_hand = any(c and c.id in (EXPLORER, LILLIE) for c in (my_state(obs).hand or []) if c)
            if metal_total <= 2 and not has_cind and not draw_in_hand:
                best_stall = -500
                stall_reason = "save Boss"
                for target in opp_bench_pokemon(obs):
                    te = energy_count(target)
                    cd = CARD_DB.get(target.id)
                    rc = cd.retreatCost if cd else 0
                    min_atk = 99
                    if cd and cd.attacks:
                        for aid in cd.attacks:
                            atk = ALL_ATTACKS.get(aid)
                            if atk:
                                min_atk = min(min_atk, len(atk.energies))
                    if min_atk == 99:
                        min_atk = 1
                    ss = 4000 + rc * 1000 + min_atk * 500 - te * 800
                    if ss > best_stall:
                        best_stall = ss
                        stall_reason = "Boss stall"
                return best_stall, stall_reason
        return best_score, best_reason

    return 1000, "generic play"


def score_evolve(obs, opt):
    card = option_card(obs, opt)
    target = option_target(obs, opt)
    cid = card.id if card else None
    tid = target.id if target else None
    if cid == ARCHALUDON_EX and tid == DURALUDON:
        target_is_active = opt.inPlayArea == AreaType.ACTIVE
        mc = metal_in_discard(obs)
        if target_is_active:
            if energy_count(target) >= 3 and not has_in_play(obs, ARCHALUDON_EX):
                return 17000, "evolve Active 3-energy Duraludon"
            if mc >= 2:
                return 28000 + mc * 2000, "evolve Active Duraludon"
            if mc == 1:
                return 8000, "delay Active evolve: 1 Metal"
            return -500, "hold: no Metal in discard"
        if mc >= 2:
            return 14000 + mc * 1000, "evolve Bench Duraludon"
        return -1000, "hold: evolve Active first"
    return 10000, "generic evolution"


def attach_target_score(obs, target, area):
    if target is None:
        return 0
    cid = target.id
    e = energy_count(target)

    if e >= 3:
        return -5000
    if cid == CINDERACE and e >= 1:
        return -3000

    score = 0
    if cid == CINDERACE:
        score = 3000
        if e == 0:
            score += 7000 + (12000 if area == AreaType.ACTIVE else 5000)
    elif cid in {DURALUDON, ARCHALUDON_EX}:
        score = 6000 if cid == ARCHALUDON_EX else 5500
        score += {2: 12000, 1: 7000, 0: 4000}.get(e, -1000)
        score += 1000 if area == AreaType.ACTIVE else 500
    else:
        score = 1000 + (1000 if e == 0 else 0)

    # HP-based adjustment
    if target.hp > 0:
        max_hp = getattr(target, "maxHp", target.hp)
        ratio = target.hp / max_hp if max_hp > 0 else 1
        if ratio <= 0.25:
            score -= 1500
        elif ratio <= 0.50:
            score -= 500
        else:
            score += min(1000, target.hp // 40 * 100)
    return score


def score_attach(obs, opt):
    card = option_card(obs, opt)
    target = option_target(obs, opt)
    cid = card.id if card else None
    tid = target.id if target else None

    if cid == HERO_CAPE:
        if tid == ARCHALUDON_EX and target and not has_tool(target):
            return 11000, "Hero's Cape on Archaludon ex"
        if tid == DURALUDON and target and not has_tool(target) and energy_count(target) >= 1:
            return 8000, "Hero's Cape on Duraludon"
        return -1000, "save Hero's Cape"

    if cid != METAL_ENERGY:
        return -500, "skip non-Metal"
    if obs.current.energyAttached:
        return -1000, "already attached"

    return attach_target_score(obs, target, opt.inPlayArea), "attach Metal"


def score_retreat(obs, opt):
    active = active_pokemon(obs)
    if active and active.id == ARCHALUDON_EX and has_tool(active) and active.hp > 200:
        return -5000, "don't retreat HP400 tank"
    route = archaludon_ex_attack_route(obs)
    if route and route["needs_retreat"]:
        return 13000, "retreat to attack-ready ex"
    return -100, "avoid retreat"


_MAIN_DISPATCH = {
    OptionType.PLAY: score_play, OptionType.EVOLVE: score_evolve,
    OptionType.ATTACH: score_attach, OptionType.RETREAT: score_retreat,
}


def score_option(obs, opt):
    ctx = obs.select.context

    if ctx in {SelectContext.IS_FIRST, SelectContext.MULLIGAN,
               SelectContext.SETUP_ACTIVE_POKEMON, SelectContext.SETUP_BENCH_POKEMON}:
        return score_setup(obs, opt)

    if opt.type in {OptionType.YES, OptionType.NO}:
        if ctx == SelectContext.IS_FIRST:
            return score_setup(obs, opt)
        if ctx == SelectContext.ACTIVATE:
            return (100000, "Explosiveness") if opt.type == OptionType.YES else (-100000, "never decline")
        return (1, "yes") if opt.type == OptionType.YES else (0, "no")

    if opt.type == OptionType.NUMBER:
        return (opt.number or 0), "number"

    if ctx == SelectContext.MAIN:
        fn = _MAIN_DISPATCH.get(opt.type)
        if fn:
            score, reason = fn(obs, opt)
        elif opt.type == OptionType.ABILITY:
            score, reason = 1, "ability"
        elif opt.type == OptionType.ATTACK:
            score, reason = best_attack_damage(obs, opt.attackId), "attack"
        elif opt.type == OptionType.END:
            score, reason = 0, "end turn"
        else:
            score, reason = 500, "generic MAIN"
    elif ctx == SelectContext.TO_HAND:
        score, reason = score_to_hand(obs, opt)
    elif ctx in {SelectContext.DISCARD, SelectContext.DISCARD_CARD_OR_ATTACHED_CARD}:
        score, reason = score_discard(obs, opt)
    elif ctx in {SelectContext.ATTACH_TO, SelectContext.TO_FIELD, SelectContext.TO_BENCH,
                 SelectContext.ATTACH_FROM, SelectContext.SWITCH, SelectContext.TO_ACTIVE,
                 SelectContext.HEAL, SelectContext.DAMAGE}:
        score, reason = score_target(obs, opt)
    elif ctx == SelectContext.ATTACK:
        score, reason = best_attack_damage(obs, opt.attackId), "attack"
    elif opt.type == OptionType.CARD:
        score, reason = score_to_hand(obs, opt)
    elif opt.type == OptionType.ENERGY:
        score, reason = 1000, "energy"
    elif opt.type == OptionType.END:
        score, reason = 0, "end"
    else:
        score, reason = 100, "fallback"

    return apply_overrides(obs, opt, score, reason)


def score_to_hand(obs, opt):
    card = option_card(obs, opt)
    cid = card.id if card else opt.cardId
    ids = hand_ids(obs)
    effect = getattr(obs.select, "effect", None)
    effect_id = effect.id if effect else None

    if effect_id == EXPLORER:
        has_ready = any(p and p.id in (DURALUDON, ARCHALUDON_EX) and energy_count(p) >= 3
                        for p in all_my_pokemon(obs))
        metal_in_hand = sum(1 for c in (my_state(obs).hand or []) if c and c.id == METAL_ENERGY)

        if cid == HERO_CAPE:
            has_target = any(p.id == ARCHALUDON_EX and not has_tool(p) for p in all_my_pokemon(obs))
            return (27000 if has_target else 22000), "Explorer: Hero's Cape"
        if cid == METAL_ENERGY:
            if has_ready or metal_in_hand > 0:
                return 0, "Explorer: skip energy"
            if getattr(opt, 'index', 0) == _first_option_index(obs, METAL_ENERGY):
                return 25000, "Explorer: take 1st energy"
            return 0, "Explorer: skip 2nd energy"
        if cid == ARCHALUDON_EX and need_archaludon(obs):
            return 20000, "Explorer: take Archaludon ex"
        if cid == DURALUDON and need_duraludon(obs):
            return 18000, "Explorer: take Duraludon"
        if cid == RELICANTH and not has_in_play(obs, RELICANTH) and RELICANTH not in ids:
            return 15000, "Explorer: take Relicanth"
        sup_count = sum(1 for c in (my_state(obs).hand or []) if c and c.id in (EXPLORER, LILLIE))
        if cid in (EXPLORER, LILLIE) and sup_count == 0:
            return 12000, "Explorer: take supporter"
        return 0, "Explorer: let discard"

    dura_ex_count = count_in_play(obs, DURALUDON) + count_in_play(obs, ARCHALUDON_EX)
    if cid == DURALUDON and DURALUDON not in ids and dura_ex_count <= 1:
        return 22000, "take Duraludon: backup"
    if cid == ARCHALUDON_EX and need_archaludon(obs):
        return 20000, "take Archaludon ex"
    if cid == DURALUDON and need_duraludon(obs):
        return 18000, "take Duraludon"
    if cid == CINDERACE:
        return -2000, "skip Cinderace"
    if cid == RELICANTH and not has_in_play(obs, RELICANTH):
        return 9000, "take Relicanth"
    if cid == METAL_ENERGY:
        return 8000, "take Metal Energy"
    if cid == EXPLORER and not obs.current.supporterPlayed:
        return 7500, "take Explorer"
    if cid == LILLIE and not obs.current.supporterPlayed:
        return 6500, "take Lillie"
    if cid == HERO_CAPE:
        has_target = any(p.id == ARCHALUDON_EX and not has_tool(p) for p in all_my_pokemon(obs))
        return (6000, "take Hero's Cape") if has_target else (1000, "generic take")
    if cid == FULL_METAL_LAB:
        return 5000, "take Full Metal Lab"
    if cid == BOSS:
        return 2500, "take Boss"
    return 1000, "generic take"


def score_discard(obs, opt):
    card = option_card(obs, opt)
    cid = card.id if card else opt.cardId
    ids = hand_ids(obs)
    mt = metal_in_discard(obs)
    effect = getattr(obs.select, "effect", None)
    effect_id = effect.id if effect else None

    if effect_id == ULTRA_BALL:
        mh = ids.count(METAL_ENERGY)
        if cid == METAL_ENERGY:
            if mt < 2 and mh >= 1:
                if getattr(opt, 'index', None) == _first_option_index(obs, METAL_ENERGY):
                    return 20000, "UB: 1st Metal"
                return 8000, "UB: 2nd Metal"
            return 8000, "UB: Metal"
        if cid == CINDERACE:
            return (18000, "UB: Cinderace") if (mt >= 2 or mh == 0) else (14000, "UB: Cinderace")
        draw_count = ids.count(LILLIE) + ids.count(EXPLORER)
        if cid in (LILLIE, EXPLORER) and draw_count >= 2:
            return (12000 if cid == LILLIE else 11000), "UB: surplus supporter"
        if cid == ULTRA_BALL and ids.count(ULTRA_BALL) > 1:
            return 10000, "UB: duplicate"
        if cid in (LILLIE, EXPLORER) and draw_count <= 1:
            return -3000, "UB: keep last supporter"

    if cid == METAL_ENERGY:
        if mt < 2:
            return 15000, "discard Metal"
        return (12000, "discard extra Metal") if ids.count(METAL_ENERGY) > 1 else (-1000, "keep last Metal")
    if cid == CINDERACE:
        return 10000, "discard Cinderace"
    if cid in {BOSS, FULL_METAL_LAB, POKEGEAR}:
        return 8500, "discard utility"
    if cid in {LILLIE, EXPLORER} and ids.count(cid) > 1:
        return 8000, "discard duplicate supporter"
    if cid == RELICANTH and (has_in_play(obs, RELICANTH) or ids.count(RELICANTH) > 1):
        return 6500, "discard extra Relicanth"
    if cid == ARCHALUDON_EX:
        return -5000, "keep Archaludon ex"
    if cid == DURALUDON:
        return -4000, "keep Duraludon"
    return 1000, "generic discard"


def score_target(obs, opt):
    card = option_card(obs, opt)
    cid = card.id if card else opt.cardId
    ctx = obs.select.context

    if ctx == SelectContext.ATTACH_TO:
        return (5000, "Metal") if cid == METAL_ENERGY else (1000, "attach")

    if ctx == SelectContext.ATTACH_FROM:
        if card and energy_count(card) >= 3:
            return -5000, "skip: 3+ energy"
        if card and cid == CINDERACE and energy_count(card) >= 1:
            return -3000, "skip: Cinderace ready"
        return attach_target_score(obs, card, opt.area), "effect attach"

    if ctx in {SelectContext.TO_FIELD, SelectContext.TO_BENCH}:
        if cid == ARCHALUDON_EX:
            return 18000, "target Archaludon ex"
        if cid == DURALUDON:
            return 16000, "target Duraludon"
        if cid == CINDERACE:
            return 3000, "avoid Cinderace"

    if ctx == SelectContext.HEAL:
        return (20000 + damage_on(card), "heal Archaludon ex") if cid == ARCHALUDON_EX else (damage_on(card), "heal")

    if ctx in {SelectContext.SWITCH, SelectContext.TO_ACTIVE}:
        yi = obs.current.yourIndex
        pi = getattr(opt, 'playerIndex', yi)
        if pi != yi and card:
            # vs Hop: prioritize Snorlax (remove Extra Helpings)
            if detect_matchup(obs) == "hop" and cid == HOP_SNORLAX and card:
                active = active_pokemon(obs)
                e = energy_count(card)
                tools = len(getattr(card, 'tools', None) or [])
                if active and active.id == CINDERACE:
                    # Cinderace: pull the least mobile Snorlax (low energy, no tools, high HP)
                    return 30000 - e * 100 - tools * 50 + card.hp, "Boss: Snorlax (immobile target)"
                else:
                    # Archaludon: pull the most threatening Snorlax (high energy, tools, high HP)
                    return 30000 + e * 100 + tools * 50 + card.hp, "Boss: Snorlax (biggest threat)"
            pv = prize_value(card)
            te = energy_count(card)
            killable = any(effective_damage(a["damage"], card) >= card.hp
                           for a in planned_archaludon_attacks(obs))
            if killable:
                return 20000 + pv * 3000 + te * 100, "Boss: KO"
            return 5000 + pv * 1000 + te * 200, "Boss: drag"
        if cid == CINDERACE:
            return 16000, "promote Cinderace (retreat 0)"
        if cid == ARCHALUDON_EX:
            return 15000, "promote Archaludon ex"
        if cid == DURALUDON:
            return 8000, "promote Duraludon"
        return 1000, "generic promote"

    if ctx == SelectContext.DAMAGE:
        hp = getattr(card, "hp", 999) if card else 999
        return 10000 - hp, "damage: lowest HP"

    return 1000, "generic target"


# ── Choose & Agent ──

def choose_options(obs):
    scored = []
    for i, opt in enumerate(obs.select.option):
        try:
            score, reason = score_option(obs, opt)
        except Exception as e:
            score, reason = -999999, f"error {type(e).__name__}: {e}"
        scored.append((score, i, reason))

    scored.sort(key=lambda x: (x[0], -x[1]), reverse=True)

    selected = []
    for score, i, reason in scored:
        if len(selected) >= obs.select.maxCount:
            break
        if score < 0 and len(selected) >= obs.select.minCount:
            continue
        selected.append(i)

    if len(selected) < obs.select.minCount:
        selected = [i for _, i, _ in scored[:obs.select.minCount]]

    return selected


def agent(obs_dict):
    obs = to_observation_class(obs_dict)
    if obs.select is None:
        global _opp_last_attack_id, _cur_turn_logs
        _opp_last_attack_id = None
        _cur_turn_logs.clear()
        return read_deck_csv()
    _update_opp_attack_tracking(obs)
    if not obs.select.option:
        return []
    try:
        return choose_options(obs)
    except Exception:
        return random.sample(list(range(len(obs.select.option))), obs.select.maxCount)


In [ ]:
%%writefile orig_alakazam.py
import os
import sys
from collections import defaultdict

from cg.api import AreaType, CardType, EnergyType, Observation, SelectContext, OptionType, Card, Pokemon, all_card_data, to_observation_class

"""
Alakazam Deck
This deck uses Alakazam's Powerful Hand attack (20 damage per card in hand)
with a draw engine built around Kadabra/Alakazam Psychic Draw, Dudunsparce's
Run Away Draw, and Fezandipiti ex's Flip the Script.
"""

# Load deck.csv in the dataset
file_path = "deck.csv"
if not os.path.exists(file_path):
    file_path = "/kaggle_simulations/agent/" + file_path
with open(file_path, "r") as file:
    csv = file.read().split("\n")
my_deck = []
for i in range(60):
    my_deck.append(int(csv[i]))

# Fetch card metadata database and create an ID-to-Card lookup table
all_card = all_card_data()
card_table = {c.cardId: c for c in all_card}

# Decklist
Abra = 741              # x4
Kadabra = 742            # x4
Alakazam = 743           # x3
Dunsparce = 305          # x3
Dudunsparce = 66         # x2
Fezandipiti_ex = 140     # x1
Genesect = 142           # x1
Psyduck = 858            # x1
Shaymin = 343            # x1
Rare_Candy = 1079        # x3
Enhanced_Hammer = 1081   # x3
Buddy_Buddy_Poffin = 1086  # x4
Night_Stretcher = 1097   # x1
Sacred_Ash = 1129        # x1
Poke_Pad = 1152          # x4
Lucky_Helmet = 1156      # x3
Boss_Orders = 1182       # x2
Hilda = 1225             # x4
Dawn = 1231              # x4
Battle_Cage = 1264       # x4
Basic_Psychic_Energy = 5   # x2
Telepath_Psychic_Energy = 19  # x4
Enriching_Energy = 13    # x1  (ACE SPEC)

# Opponent card IDs to watch for
Duskull = 131
Slowpoke_IDs = (162, 327)
Froakie_IDs = (33, 945)
Wellspring_Mask_Ogerpon_ex = 108
N_Darumaka = 257
Dreepy = 119
Drakloak = 120
Dragapult_ex = 121
Mist_Energy = 11
Rock_Fighting_Energy = 20

# Attack IDs
ATTACK_TELEPORTATION = 1070   # Abra: 10 dmg, cost {P}
ATTACK_SUPER_PSY_BOLT = 1071  # Kadabra: 30 dmg, cost {P}
ATTACK_POWERFUL_HAND = 1072   # Alakazam: 20 per card in hand, cost {P}

# Card ID sets
ABRA_LINE = {Abra, Kadabra, Alakazam}
DUNSPARCE_LINE = {Dunsparce, Dudunsparce}
PSYCHIC_ENERGY_IDS = {Basic_Psychic_Energy, Telepath_Psychic_Energy}

pre_turn = 0
ability_used_dudunsparce = False
ability_used_fezandipiti = False


def get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:
    ps = obs.current.players[player_index]
    match area:
        case AreaType.DECK:
            return obs.select.deck[index]
        case AreaType.HAND:
            return ps.hand[index]
        case AreaType.DISCARD:
            return ps.discard[index]
        case AreaType.ACTIVE:
            return ps.active[index]
        case AreaType.BENCH:
            return ps.bench[index]
        case AreaType.PRIZE:
            return ps.prize[index]
        case AreaType.STADIUM:
            return obs.current.stadium[index]
        case AreaType.LOOKING:
            return obs.current.looking[index]
        case _:
            return None


def prize_count(pokemon: Pokemon) -> int:
    data = card_table[pokemon.id]
    count = 3 if data.megaEx else 2 if data.ex else 1
    for card in pokemon.energyCards:
        if card.id == 12:  # Legacy Energy
            count -= 1
    for card in pokemon.tools:
        if card.id == 1172 and "Lillie" in data.name:
            count -= 1
    return max(0, count)


def count_special_defense_energies(pokemon: Pokemon) -> int:
    cnt = 0
    for ec in pokemon.energyCards:
        if ec.id == Mist_Energy or ec.id == Rock_Fighting_Energy:
            cnt += 1
    return cnt



# Crash-safety and submission diagnostics. The public policy below is preserved as
# _policy_agent(); the submitted agent() wrapper normalizes its option indices and
# falls back legally on every exception.
_DIAG = defaultdict(int)


def diag_reset() -> None:
    _DIAG.clear()


def diag_snapshot() -> dict:
    total = max(1, _DIAG.get("decisions", 0))
    out = dict(_DIAG)
    out["fallback_rate"] = (_DIAG.get("policy_fallback", 0) + _DIAG.get("obs_fallback", 0)) / total
    return out


def _legal_fallback(select) -> list[int]:
    try:
        n = len(select.option)
        if n == 0 or select.maxCount <= 0:
            return []
        min_count = max(0, min(select.minCount, n))
        max_count = max(min_count, min(select.maxCount, n))
        return list(range(min_count if min_count > 0 else min(1, max_count)))
    except Exception:
        return []


def _normalize_ordered_indices(indices, select) -> list[int]:
    try:
        n = len(select.option)
        if n == 0 or select.maxCount <= 0:
            return []
        min_count = max(0, min(select.minCount, n))
        max_count = max(min_count, min(select.maxCount, n))
        out = []
        seen = set()
        for idx in indices or []:
            if not isinstance(idx, int) or idx in seen or not (0 <= idx < n):
                continue
            seen.add(idx)
            out.append(idx)
            if len(out) >= max_count:
                break
        if len(out) < min_count:
            for idx in range(n):
                if idx not in seen:
                    out.append(idx)
                    seen.add(idx)
                if len(out) >= min_count:
                    break
        return out[:max_count]
    except Exception:
        return _legal_fallback(select)


def _policy_agent(obs_dict: dict) -> list[int]:
    obs = to_observation_class(obs_dict)
    if obs.select is None:
        return my_deck

    state = obs.current
    select = obs.select
    context = select.context
    my_index = state.yourIndex
    my_state = state.players[my_index]
    op_state = state.players[1 - my_index]
    my_prize_count = len(my_state.prize)

    global pre_turn, ability_used_dudunsparce, ability_used_fezandipiti
    if pre_turn != state.turn:
        pre_turn = state.turn
        ability_used_dudunsparce = False
        ability_used_fezandipiti = False

    # ---- Count cards on field / hand / discard ----
    field_counts = defaultdict(int)
    hand_counts = defaultdict(int)
    discard_counts = defaultdict(int)

    my_field = []  # (field_index, pokemon) where 0=active, 1..=bench
    for card in my_state.active:
        if card is not None:
            field_counts[card.id] += 1
            my_field.append((0, card))
    for idx, card in enumerate(my_state.bench):
        if card is not None:
            field_counts[card.id] += 1
            my_field.append((idx + 1, card))

    for card in my_state.hand:
        hand_counts[card.id] += 1

    for card in my_state.discard:
        discard_counts[card.id] += 1

    abra_line_on_field = field_counts[Abra] + field_counts[Kadabra] + field_counts[Alakazam]
    dunsparce_line_on_field = field_counts[Dunsparce] + field_counts[Dudunsparce]

    # ---- Opponent field analysis ----
    op_all_pokemon = []
    for card in op_state.active:
        if card is not None:
            op_all_pokemon.append(card)
    for card in op_state.bench:
        if card is not None:
            op_all_pokemon.append(card)

    op_has_duskull = any(p.id == Duskull for p in op_all_pokemon)
    op_has_water_threat = any(
        p.id in Slowpoke_IDs or p.id in Froakie_IDs
        or p.id == Wellspring_Mask_Ogerpon_ex or p.id == N_Darumaka
        for p in op_all_pokemon
    )
    op_has_dragapult_line = any(
        p.id in (Dreepy, Drakloak, Dragapult_ex) for p in op_all_pokemon
    )

    # Detect if opponent has used ACE SPEC
    op_used_ace_spec = False
    for log in obs.logs:
        if hasattr(log, 'cardId') and log.cardId is not None:
            cd = card_table.get(log.cardId)
            if cd and cd.aceSpec and hasattr(log, 'playerIndex') and log.playerIndex == (1 - my_index):
                op_used_ace_spec = True

    stadium_id = 0
    for card in state.stadium:
        stadium_id = card.id

    bench_count = len(my_state.bench)
    bench_max = my_state.benchMax
    bench_free = bench_max - bench_count

    # ---- Active pokemon info ----
    active_pokemon = my_state.active[0] if my_state.active else None
    active_id = active_pokemon.id if active_pokemon else -1
    active_has_psychic = False
    if active_pokemon:
        for ec in active_pokemon.energyCards:
            if ec.id in PSYCHIC_ENERGY_IDS:
                active_has_psychic = True
                break

    # ---- Opponent active info ----
    op_active = op_state.active[0] if op_state.active else None
    op_active_hp = op_active.hp if op_active else 9999

    # ---- Estimate Powerful Hand damage range ----
    hand_size = len(my_state.hand) if my_state.hand else my_state.handCount

    def estimate_hand_increase():
        """Returns (min_increase, max_increase) of hand size this turn from draw effects."""
        min_inc = 0
        max_inc = 0
        for _, p in my_field:
            if p.id == Abra and hand_counts[Kadabra] > 0:
                max_inc += 1  # evolve Kadabra: hand -1, draw +2 = net +1
            elif p.id == Abra and hand_counts[Rare_Candy] > 0 and hand_counts[Alakazam] > 0:
                max_inc += 1  # Rare Candy + Alakazam: hand -2, draw +3 = net +1
            elif p.id == Kadabra and hand_counts[Alakazam] > 0:
                max_inc += 2  # evolve Alakazam: hand -1, draw +3 = net +2
            elif p.id == Dunsparce and hand_counts[Dudunsparce] > 0:
                max_inc += 1  # evolve: hand -1, ability draw +2 = net +1
            elif p.id == Dudunsparce:
                if not ability_used_dudunsparce:
                    max_inc += 3  # Run Away Draw
            elif p.id == Fezandipiti_ex:
                if not ability_used_fezandipiti:
                    max_inc += 3  # Flip the Script
        if hand_counts[Fezandipiti_ex] > 0 and bench_free > 0 and field_counts[Fezandipiti_ex] == 0:
            max_inc += 2  # play -1, ability +3 = net +2

        # Supporter (only 1 can be used)
        supporter_options = []
        if not state.supporterPlayed:
            if hand_counts[Hilda] > 0:
                supporter_options.append(1)   # play -1, search +2 = net +1
            if hand_counts[Dawn] > 0:
                supporter_options.append(2)   # play -1, search +3 = net +2
            if hand_counts[Boss_Orders] > 0:
                supporter_options.append(-1)  # play -1 = net -1
        if supporter_options:
            max_inc += max(supporter_options)

        # Enriching Energy attach: hand -1, draw +4 = net +3
        if hand_counts[Enriching_Energy] > 0 and not state.energyAttached:
            if active_id == Alakazam and active_has_psychic:
                max_inc += 3
        return min_inc, max_inc

    min_hand_inc, max_hand_inc = estimate_hand_increase()
    max_hand_size = hand_size + max_hand_inc
    min_hand_size = hand_size + min_hand_inc
    max_damage = max_hand_size * 20
    min_damage = min_hand_size * 20

    # ---- Target selection for attack ----
    target_idx = -1       # 0 = active, 1.. = bench
    target_pokemon = None
    target_use_boss = False
    target_can_kill = False
    target_prize_gain = 0
    target_hammer_needed = 0
    use_kadabra_finish = False

    if state.turn >= 2 and op_active is not None:
        # Check Kadabra finisher: opponent active HP <= 30
        if op_active_hp <= 30 and (field_counts[Kadabra] >= 1 or active_id == Kadabra):
            target_idx = 0
            target_pokemon = op_active
            target_use_boss = False
            target_can_kill = True
            target_prize_gain = prize_count(op_active)
            use_kadabra_finish = True
        else:
            # Evaluate all opponent pokemon
            all_op = [(0, op_active)]
            for bi, bp in enumerate(op_state.bench):
                if bp is not None:
                    all_op.append((bi + 1, bp))

            candidates = []
            for oi, pkmn in all_op:
                pz = prize_count(pkmn)
                sp_e = count_special_defense_energies(pkmn)
                eff_max_dmg = max_damage
                hm_need = 0
                if sp_e > 0:
                    if hand_counts[Enhanced_Hammer] >= sp_e:
                        hm_need = sp_e
                        eff_max_dmg = (max_hand_size - hm_need) * 20
                    else:
                        eff_max_dmg = 0
                ck = pkmn.hp <= eff_max_dmg and eff_max_dmg > 0
                candidates.append((oi, pkmn, pz, ck, hm_need))

            # Priority 1: kill wins the game
            win_cands = [(oi, pk, pz, ck, hm) for oi, pk, pz, ck, hm in candidates if ck and my_prize_count <= pz]
            if win_cands:
                # Among winners, prefer active (no boss needed), then highest HP
                best = min(win_cands, key=lambda x: (0 if x[0] == 0 else 1, -x[1].hp))
                target_idx, target_pokemon, target_prize_gain, target_can_kill, target_hammer_needed = best
                target_use_boss = target_idx != 0
            else:
                # Priority 2: killable target with most prizes
                killable = [(oi, pk, pz, ck, hm) for oi, pk, pz, ck, hm in candidates if ck]
                if killable:
                    best = max(killable, key=lambda x: (x[2], x[1].hp))
                    target_idx, target_pokemon, target_prize_gain, target_can_kill, target_hammer_needed = best
                    target_use_boss = target_idx != 0
                else:
                    # Priority 3: just hit active
                    target_idx = 0
                    target_pokemon = op_active
                    target_use_boss = False
                    target_can_kill = False
                    target_prize_gain = 0

    # Should we use Dudunsparce's ability?
    need_dudunsparce_draw = False
    if target_pokemon is not None and target_can_kill:
        needed = target_pokemon.hp
        current_dmg = (hand_size - target_hammer_needed) * 20
        if current_dmg < needed:
            need_dudunsparce_draw = True

    # Do we need to attach energy to the active to retreat?
    need_retreat_energy = False
    if active_pokemon is not None and state.turn >= 2:
        active_is_attacker = (active_id == Alakazam and active_has_psychic) or (use_kadabra_finish and active_id == Kadabra)
        if not active_is_attacker:
            # Check if there's a better attacker on bench
            has_bench_attacker = False
            if use_kadabra_finish and field_counts[Kadabra] >= 1 and active_id != Kadabra:
                has_bench_attacker = True
            elif field_counts[Alakazam] >= 1 and active_id != Alakazam:
                has_bench_attacker = True
            elif field_counts[Kadabra] >= 1 and active_id != Kadabra:
                has_bench_attacker = True
            if has_bench_attacker:
                retreat_cost = card_table[active_pokemon.id].retreatCost
                active_energy_count = len(active_pokemon.energies)
                if active_energy_count < retreat_cost:
                    need_retreat_energy = True

    # Do we need Fezandipiti ex's Flip the Script to kill the target?
    fez_hand_contribution = 0
    if field_counts[Fezandipiti_ex] >= 1 and not ability_used_fezandipiti:
        fez_hand_contribution = 3
    elif hand_counts[Fezandipiti_ex] > 0 and bench_free > 0 and field_counts[Fezandipiti_ex] == 0:
        fez_hand_contribution = 2  # play -1, ability +3 = net +2
    need_fezandipiti_draw = False
    if target_pokemon is not None and target_can_kill and fez_hand_contribution > 0:
        max_damage_without_fez = (max_hand_size - fez_hand_contribution - target_hammer_needed) * 20
        if max_damage_without_fez < target_pokemon.hp:
            need_fezandipiti_draw = True

    # Also allow Fezandipiti if drawing could find key enablers (Boss, Rare Candy, Alakazam, Energy)
    need_fezandipiti_for_setup = False
    if target_pokemon is not None and target_can_kill and fez_hand_contribution > 0 and not need_fezandipiti_draw:
        # Missing Boss's Orders for bench target
        missing_boss = (target_use_boss and hand_counts[Boss_Orders] == 0
                        and not state.supporterPlayed)
        # Check if we have a ready attacker (Alakazam with psychic energy)
        has_ready_attacker = (active_id == Alakazam and active_has_psychic)
        if not has_ready_attacker:
            for _, p in my_field:
                if p.id == Alakazam and any(ec.id in PSYCHIC_ENERGY_IDS for ec in p.energyCards):
                    has_ready_attacker = True
                    break
        missing_attacker = False
        missing_energy = False
        if not has_ready_attacker:
            # Can we set up Alakazam this turn?
            can_evolve_to_alakazam = (field_counts[Kadabra] >= 1 and hand_counts[Alakazam] >= 1)
            can_rare_candy_alakazam = (field_counts[Abra] >= 1 and hand_counts[Rare_Candy] >= 1
                                       and hand_counts[Alakazam] >= 1)
            if not can_evolve_to_alakazam and not can_rare_candy_alakazam:
                # Missing evolution pieces
                if field_counts[Kadabra] >= 1 and hand_counts[Alakazam] == 0:
                    missing_attacker = True
                elif field_counts[Abra] >= 1 and (hand_counts[Rare_Candy] == 0 or hand_counts[Alakazam] == 0):
                    missing_attacker = True
            # Check if energy is available for the attacker
            energy_in_hand = (hand_counts[Basic_Psychic_Energy] + hand_counts[Telepath_Psychic_Energy]
                              + hand_counts[Enriching_Energy])
            if not state.energyAttached and energy_in_hand == 0:
                has_energized = any(
                    p.id in ABRA_LINE and any(ec.id in PSYCHIC_ENERGY_IDS for ec in p.energyCards)
                    for _, p in my_field
                )
                if not has_energized:
                    missing_energy = True
        if missing_boss or missing_attacker or missing_energy:
            need_fezandipiti_for_setup = True

    # Deck safety: don't let deck count drop to <= prize count unless winning this turn
    can_win_this_turn = target_can_kill and my_prize_count <= target_prize_gain
    deck_count = my_state.deckCount
    # safe_draws: max cards we can draw from deck while keeping deck > prize count
    # We also need 1 card for the draw at start of next turn
    safe_draws = deck_count - my_prize_count - 1 if not can_win_this_turn else 999

    # ---- Score each option ----
    scores = []
    for o in select.option:
        score = 0

        if o.type == OptionType.NUMBER:
            score = o.number

        elif o.type == OptionType.YES:
            score = 1

        elif o.type == OptionType.CARD:
            card = get_card(obs, o.area, o.index, o.playerIndex)
            if card is None:
                scores.append(score)
                continue
            energy_count = len(card.energies) if isinstance(card, Pokemon) else 0

            if context == SelectContext.SWITCH or context == SelectContext.TO_ACTIVE:
                if o.playerIndex == my_index:
                    if card.id == Alakazam:
                        score += 100 + energy_count * 10
                    elif card.id == Kadabra:
                        score += 90 if (op_active_hp <= 30) else 30
                    elif card.id == Abra:
                        score += 10
                    elif card.id in DUNSPARCE_LINE:
                        score += 5
                    else:
                        score += 1
                else:
                    if target_use_boss and target_pokemon is not None:
                        if o.index == target_idx - 1:
                            score += 100

            elif context == SelectContext.SETUP_ACTIVE_POKEMON:
                if card.id == Abra:
                    score = 10
                elif card.id == Dunsparce:
                    score = 5
                elif card.id == Psyduck:
                    score = 2
                elif card.id == Shaymin:
                    score = 1

            elif context == SelectContext.SETUP_BENCH_POKEMON:
                if card.id == Abra:
                    cur = field_counts[Abra] + field_counts[Kadabra] + field_counts[Alakazam]
                    score = 200 if cur == 0 else 100 + (3 - cur) * 10
                elif card.id == Dunsparce:
                    score = 150 if dunsparce_line_on_field == 0 else 50

            elif context == SelectContext.TO_HAND:
                score = 200 - hand_counts.get(card.id, 0) * 50
                if card.id == Dudunsparce:
                    score += 80 if (field_counts[Dunsparce] >= 1 and field_counts[Dudunsparce] == 0) else -50
                elif card.id == Kadabra:
                    score += 70 if field_counts[Abra] >= 1 else -20
                elif card.id == Alakazam:
                    score += 60 if (field_counts[Kadabra] >= 1 or field_counts[Abra] >= 1) else -20
                elif card.id == Abra:
                    score += 50 if abra_line_on_field < 3 else -50
                elif card.id == Dunsparce:
                    score += 40 if dunsparce_line_on_field < 2 else -50
                elif card.id in PSYCHIC_ENERGY_IDS:
                    score += 30 if not state.energyAttached else -10
                elif card.id == Enriching_Energy:
                    score += 20
                elif card.id == Rare_Candy:
                    score += 40 if field_counts[Abra] >= 1 else -10

            elif context == SelectContext.ATTACH_FROM:
                if isinstance(card, Pokemon):
                    if need_retreat_energy and o.area == AreaType.ACTIVE:
                        score = 150  # Must attach to active to retreat
                    elif len(card.energyCards) >= 1:
                        score = -1  # Don't attach 2+ energy to the same pokemon
                    elif card.id in ABRA_LINE:
                        score = 100
                        if card.id == Alakazam:
                            score += 20
                        elif card.id == Kadabra:
                            score += 10
                        if o.area == AreaType.ACTIVE:
                            score += 5
                    elif card.id in DUNSPARCE_LINE:
                        score = 50
                    else:
                        score = 10

            elif context == SelectContext.TO_BENCH:
                if card.id == Abra:
                    score = 100
                elif card.id == Dunsparce:
                    score = 80
                elif card.id == Psyduck:
                    if op_has_duskull:
                        score = 60
                    else:
                        score = -1
                elif card.id == Shaymin:
                    if op_has_water_threat:
                        score = 40
                    else:
                        score = -1

            elif context == SelectContext.TO_DECK:
                if card.id in ABRA_LINE:
                    score = 100
                elif card.id in DUNSPARCE_LINE:
                    score = 50
                else:
                    score = 10

        elif o.type == OptionType.PLAY:
            card = get_card(obs, AreaType.HAND, o.index, my_index)
            data = card_table[card.id]

            if data.cardType == CardType.POKEMON:
                score = 20000
                is_early = state.turn <= 2

                if card.id == Abra:
                    if is_early:
                        score += 500
                    elif abra_line_on_field < 3:
                        score += 200
                    elif bench_free <= 1:
                        score = -1
                    else:
                        score += 50

                elif card.id == Dunsparce:
                    if dunsparce_line_on_field < 1:
                        score += 400 if is_early else 100
                    elif dunsparce_line_on_field < 2:
                        score += 50
                    else:
                        score = -1

                elif card.id == Fezandipiti_ex:
                    if need_fezandipiti_draw or need_fezandipiti_for_setup:
                        score += 80 if not is_early else 30
                    else:
                        score = -1  # Don't play unless Flip the Script is needed to kill

                elif card.id == Genesect:
                    if not op_used_ace_spec and (hand_counts[Lucky_Helmet] > 0 or hand_counts[Poke_Pad] > 0):
                        score += 100
                    else:
                        score = -1

                elif card.id == Psyduck:
                    if op_has_duskull:
                        score += 300
                    else:
                        score = -1

                elif card.id == Shaymin:
                    if op_has_water_threat:
                        score += 300
                    else:
                        score = -1

                # Keep at least 1 bench slot free
                if bench_free <= 1 and score > 0:
                    score -= 5000

            else:
                score = 10000

                if card.id == Buddy_Buddy_Poffin:
                    if safe_draws < 2:
                        score = -1  # Deck too thin (searches deck)
                    elif state.turn <= 2:
                        if abra_line_on_field < 3 or dunsparce_line_on_field < 1:
                            score = 18000
                        else:
                            score = 8000
                    else:
                        if abra_line_on_field < 3 or dunsparce_line_on_field < 2:
                            score = 15000
                        elif target_can_kill:
                            score = 8000
                        else:
                            score = -1

                elif card.id == Poke_Pad:
                    if safe_draws < 1:
                        score = -1  # Deck too thin (searches deck)
                    elif state.turn <= 2:
                        score = 17000
                    else:
                        score = 14000 if abra_line_on_field < 3 else 12000

                elif card.id == Rare_Candy:
                    if field_counts[Abra] >= 1 and hand_counts[Alakazam] >= 1 and safe_draws >= 3:
                        score = 16000
                    else:
                        score = -1

                elif card.id == Night_Stretcher:
                    dis_abra = discard_counts[Abra] + discard_counts[Kadabra] + discard_counts[Alakazam]
                    if dis_abra >= 1:
                        score = 13000
                    elif discard_counts[Basic_Psychic_Energy] + discard_counts[Telepath_Psychic_Energy] >= 1:
                        score = 11000
                    else:
                        score = -1

                elif card.id == Sacred_Ash:
                    dis_abra = discard_counts[Abra] + discard_counts[Kadabra] + discard_counts[Alakazam]
                    if dis_abra >= 2:
                        score = 13500
                    elif dis_abra >= 1:
                        score = 11000
                    else:
                        score = -1

                elif card.id == Enhanced_Hammer:
                    if target_hammer_needed > 0:
                        score = 6500
                    else:
                        # Check if any opponent pokemon has special defense energy
                        any_special = any(count_special_defense_energies(p) > 0 for p in op_all_pokemon)
                        if any_special:
                            score = 5000
                        else:
                            score = -1

                elif card.id == Lucky_Helmet:
                    score = 7000  # Will be handled via ATTACH

                elif card.id == Boss_Orders:
                    if target_use_boss and target_can_kill:
                        score = 3200
                    else:
                        score = -1

                elif card.id == Hilda:
                    if safe_draws >= 2:
                        score = 3000
                    else:
                        score = -1

                elif card.id == Dawn:
                    if safe_draws >= 3:
                        score = 3100
                    else:
                        score = -1

                elif card.id == Battle_Cage:
                    if op_has_dragapult_line:
                        score = 19000
                    elif stadium_id != 0:
                        score = 7000
                    else:
                        score = -1

        elif o.type == OptionType.ATTACH:
            card = get_card(obs, AreaType.HAND, o.index, my_index)
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)

            if card.id == Lucky_Helmet:
                score = 7000
                if pokemon.id == Genesect and not op_used_ace_spec:
                    score += 300
                elif o.inPlayArea == AreaType.ACTIVE:
                    score += 200
                else:
                    score += 50

            elif card.id in PSYCHIC_ENERGY_IDS:
                if need_retreat_energy and o.inPlayArea == AreaType.ACTIVE:
                    score = 9500  # Must attach to active to retreat
                elif len(pokemon.energyCards) >= 1:
                    score = -1  # Don't attach 2+ energy to the same pokemon
                elif pokemon.id in ABRA_LINE:
                    score = 8000
                    if pokemon.id == Alakazam:
                        score += 30
                    elif pokemon.id == Kadabra:
                        score += 20
                    elif pokemon.id == Abra:
                        score += 10
                    if o.inPlayArea == AreaType.ACTIVE:
                        score += 5
                else:
                    score = -1
                # Telepath Psychic Energy searches 2 from deck
                if card.id == Telepath_Psychic_Energy and safe_draws < 2 and score > 0:
                    score = -1

            elif card.id == Enriching_Energy:
                if need_retreat_energy and o.inPlayArea == AreaType.ACTIVE:
                    score = 9500  # Must attach to active to retreat
                elif len(pokemon.energyCards) >= 1:
                    score = -1  # Don't attach 2+ energy to the same pokemon
                elif pokemon.id in DUNSPARCE_LINE:
                    score = 8500
                    if pokemon.id == Dudunsparce:
                        score += 10
                else:
                    score = -1
                # Enriching Energy draws 4 from deck
                if card.id == Enriching_Energy and safe_draws < 4 and score > 0:
                    score = -1

        elif o.type == OptionType.EVOLVE:
            card = get_card(obs, AreaType.HAND, o.index, my_index)
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            score = 9000

            if card.id == Alakazam:
                if safe_draws < 3:
                    score = -1  # Deck too thin for Psychic Draw (3 cards)
                elif o.inPlayArea == AreaType.ACTIVE:
                    score += 200  # Active Alakazam = highest
                else:
                    score += 50  # Bench Alakazam
                score += len(pokemon.energies) * 10

            elif card.id == Kadabra:
                if safe_draws < 2:
                    score = -1  # Deck too thin for Psychic Draw (2 cards)
                else:
                    score += 100
                    if len(pokemon.energies) == 0:
                        score += 50  # Evolve non-energy Abra first
                    else:
                        score -= 20
                        if hand_counts[Rare_Candy] > 0 and hand_counts[Alakazam] > 0:
                            score -= 100  # Save energy Abra for Rare Candy -> Alakazam

            elif card.id == Dudunsparce:
                if safe_draws < 2:
                    score = -1  # Deck too thin for draw on evolve
                else:
                    score += 80

        elif o.type == OptionType.ABILITY:
            card = get_card(obs, o.area, o.index, my_index)
            if card is None:
                scores.append(score)
                continue

            if card.id == Dudunsparce:
                if need_dudunsparce_draw:
                    if safe_draws >= 3:
                        score = 30000
                    else:
                        score = -1  # Deck too thin
                else:
                    score = -1
            elif card.id == Fezandipiti_ex:
                if (need_fezandipiti_draw or need_fezandipiti_for_setup) and safe_draws >= 3:
                    score = 29000
                else:
                    score = -1  # Don't use unless needed to kill target
            elif card.id == Battle_Cage:
                score = 1
            else:
                score = 28000

        elif o.type == OptionType.RETREAT:
            if active_id == Alakazam and active_has_psychic:
                score = -1
            elif use_kadabra_finish and active_id != Kadabra and field_counts[Kadabra] >= 1:
                score = 2500  # Retreat to bring Kadabra forward for finish
            elif active_id in (Abra, Dunsparce, Dudunsparce, Psyduck, Shaymin, Genesect):
                if field_counts[Alakazam] >= 1 or field_counts[Kadabra] >= 1:
                    score = 2000
                else:
                    score = -1
            else:
                score = -1

        elif o.type == OptionType.ATTACK:
            score = 1000
            if o.attackId == ATTACK_POWERFUL_HAND:
                score += 500
            elif o.attackId == ATTACK_SUPER_PSY_BOLT:
                if op_active_hp <= 30:
                    score += 600  # Kadabra finisher
                else:
                    score += 100
            elif o.attackId == ATTACK_TELEPORTATION:
                score += 50

        scores.append(score)

    # Select in descending order of score
    desc_indices = [i for i, _ in sorted(enumerate(scores), key=lambda x: x[1], reverse=True)]

    if context == SelectContext.MAIN:
        o = select.option[desc_indices[0]]
        if o.type == OptionType.ABILITY:
            card = get_card(obs, o.area, o.index, my_index)
            if card is not None:
                if card.id == Dudunsparce:
                    ability_used_dudunsparce = True
                elif card.id == Fezandipiti_ex:
                    ability_used_fezandipiti = True

    return desc_indices[:select.maxCount]


def agent(obs_dict: dict) -> list[int]:
    try:
        if isinstance(obs_dict, dict) and obs_dict.get("select") is None:
            _DIAG["deck_returns"] += 1
            return my_deck
    except Exception:
        pass

    _DIAG["decisions"] += 1
    try:
        obs = to_observation_class(obs_dict)
        if obs.select is None:
            _DIAG["deck_returns"] += 1
            _DIAG["decisions"] -= 1
            return my_deck
        raw = _policy_agent(obs_dict)
        selection = _normalize_ordered_indices(raw, obs.select)
        if selection:
            _DIAG["policy_ok"] += 1
            return selection
        _DIAG["policy_fallback"] += 1
        return _legal_fallback(obs.select)
    except Exception:
        _DIAG["policy_fallback"] += 1
        try:
            obs = to_observation_class(obs_dict)
            if obs.select is None:
                _DIAG["deck_returns"] += 1
                _DIAG["decisions"] -= 1
                return my_deck
            return _legal_fallback(obs.select)
        except Exception:
            _DIAG["obs_fallback"] += 1
            if isinstance(obs_dict, dict) and obs_dict.get("select") is None:
                return my_deck
            return []


Their decklists, as plain CSVs (verified: every card ID resolved correctly against the real 1267-card database when tested).

In [ ]:
from pathlib import Path

Path("_orig_deck_fighting.csv").write_text("""673
673
674
674
675
675
676
676
676
677
677
677
678
678
678
678
1102
1102
1102
1102
1123
1123
1141
1141
1141
1141
1142
1142
1142
1142
1152
1152
6
1159
1182
1182
1192
1192
1192
1192
1227
1227
1227
1227
6
6
6
6
6
6
6
6
6
6
6
6
6
1182
677
1252
""")
Path("_orig_deck_A.csv").write_text("""8
8
8
8
8
8
8
8
8
8
8
57
169
169
169
169
190
190
190
190
666
666
666
666
1097
1097
1097
1121
1121
1121
1121
1122
1122
1122
1122
1147
1147
1147
1147
1152
1152
1152
1152
1159
1182
1182
1182
1185
1185
1185
1185
1213
1227
1227
1227
1227
1244
1244
1244
1244
""")
Path("_orig_deck_B.csv").write_text("""5
5
5
13
19
19
19
19
66
66
66
305
305
305
305
741
741
741
741
742
742
742
742
743
743
743
743
1079
1079
1079
1079
1081
1081
1081
1081
1086
1086
1086
1086
1097
1097
1097
1129
1152
1152
1152
1152
1182
1182
1182
1184
1225
1225
1225
1225
1231
1231
1231
1231
1264
""")
print("wrote 3 deck files")

In [ ]:
%%writefile cg_api_reference.py
from dataclasses import dataclass
from enum import IntEnum
import json
import ctypes

from .sim import lib
from .utils import to_dataclass, json_to_dataclass

#region Enums

class AreaType(IntEnum):
    DECK = 1,
    HAND = 2,
    DISCARD = 3, # Discard Pile
    ACTIVE = 4, # Active Spot
    BENCH = 5,
    PRIZE = 6,
    STADIUM = 7,
    ENERGY = 8,
    TOOL = 9,
    PRE_EVOLUTION = 10, # The pre-evolved form of the Pokémon in play.
    PLAYER = 11,
    LOOKING = 12, # The card you are looking.

class EnergyType(IntEnum):
    COLORLESS = 0,
    GRASS = 1,
    FIRE = 2,
    WATER = 3,
    LIGHTNING = 4,
    PSYCHIC = 5,
    FIGHTING = 6,
    DARKNESS = 7,
    METAL = 8,
    DRAGON = 9,
    RAINBOW = 10, # Every Types
    TEAM_ROCKET = 11, # PSYCHIC and DARKNESS 

class CardType(IntEnum):
    POKEMON = 0,
    ITEM = 1,
    TOOL = 2, # Pokémon Tool
    SUPPORTER = 3,
    STADIUM = 4,
    BASIC_ENERGY = 5,
    SPECIAL_ENERGY = 6,

class SpecialConditionType(IntEnum):
    POISON = 0,
    BURN = 1,
    SLEEP = 2,
    PARALYZE = 3,
    CONFUSE = 4,

class SelectType(IntEnum):
    MAIN = 0, # OptionType: PLAY, ATTACH, EVOLVE, ABILITY, DISCARD, RETREAT, ATTACK, END
    CARD = 1, # OptionType: CARD
    ATTACHED_CARD = 2, # OptionType: TOOL_CARD, ENERGY_CARD
    CARD_OR_ATTACHED_CARD = 3, # OptionType: CARD, TOOL_CARD, ENERGY_CARD
    ENERGY = 4, # OptionType: ENERGY
    SKILL = 5, # OptionType: SKILL
    ATTACK = 6, # OptionType: ATTACK
    EVOLVE = 7, # OptionType: EVOLVE
    COUNT = 8, # OptionType: NUMBER
    YES_NO = 9, # OptionType: YES, NO
    SPECIAL_CONDITION = 10, # OptionType: SPECIAL_CONDITION
    
class SelectContext(IntEnum):
    MAIN = 0, # Main. Main selection.
    SETUP_ACTIVE_POKEMON = 1, # Card. Select the Pokémon to put into your Active Spot during Set Up.
    SETUP_BENCH_POKEMON = 2, # Card. Select the Pokémon to put onto your Bench during Set Up.
    SWITCH = 3, # Card. Select the Pokémon to swap with the one in your Active Spot.
    TO_ACTIVE = 4, # Card. Select the Pokémon to put into your Active Spot.
    TO_BENCH = 5, # Card. Select the Pokémon to put onto your Bench.
    TO_FIELD = 6, # Card. Select the Pokémon to put into play.
    TO_HAND = 7, # Card. Select the card to add to your hand.
    DISCARD = 8, # Card. Select the card to discard.
    TO_DECK = 9, # Card. Select the card to return to your deck.
    TO_DECK_BOTTOM = 10, # Card. Select the card to return to the bottom of your deck.
    TO_PRIZE = 11, # Card. Select the card to add to your prize.
    NOT_MOVE = 12, # Card. Select the card to remain where it is.
    DAMAGE_COUNTER = 13, # Card. Select the Pokémon to place damage counters on.
    DAMAGE_COUNTER_ANY = 14, # Card. Select the Pokémon to place damage counters on using the effect that lets you place them as you like.
    DAMAGE = 15, # Card. Select the Pokémon to deal damage.
    REMOVE_DAMAGE_COUNTER = 16, # Card. Select the Pokémon to remove damage counters from.
    HEAL = 17, # Card. Select the Pokémon to heal.
    EVOLVES_FROM = 18, # Card. Select the Pokémon to evolve from.
    EVOLVES_TO = 19, # Card. Select the Pokémon to evolve into.
    DEVOLVE = 20, # Card. Select the Pokémon to devolve.
    ATTACH_FROM = 21, # Card. Select the Pokémon to attach the card to.
    ATTACH_TO = 22, # Card. Select the card to attach to the Pokémon.
    DETACH_FROM = 23, # Card. Select the Pokémon to remove the card from.
    LOOK = 24, # Card. Select the card to look at.
    EFFECT_TARGET = 25, # Card. Select the card to apply the effect to.
    DISCARD_ENERGY_CARD = 26, # AttachedCard. Select the Energy card to discard.
    DISCARD_TOOL_CARD = 27, # AttachedCard. Select the Pokémon tool to trash.
    SWITCH_ENERGY_CARD = 28, # AttachedCard. Select the energy card to replace.
    DISCARD_CARD_OR_ATTACHED_CARD = 29, # CardOrAttachedCard. Select the card to discard.
    DISCARD_ENERGY = 30, # Energy. Select the energy to discard.
    TO_HAND_ENERGY = 31, # Energy. Select the energy to return to your hand.
    TO_DECK_ENERGY = 32, # Energy. Select the energy to return to the deck.
    SWITCH_ENERGY = 33, # Energy. Select the energy to switch.
    SKILL_ORDER = 34, # Skill. Select the order of effect activation.
    ATTACK = 35, # Attack. Select the Attack to use.
    DISABLE_ATTACK = 36, # Attack. Select the Attack to disable.
    EVOLVE = 37, # Evolve. Select the Pokémon that is the evolution source and the Pokémon that is the evolution target.
    DRAW_COUNT = 38, # Count. Select how many cards to draw.
    DAMAGE_COUNTER_COUNT = 39, # Count. Select how many damage counters to place.
    REMOVE_DAMAGE_COUNTER_COUNT = 40, # Count. Select how many damage counters to remove.
    IS_FIRST = 41, # YesNo. Would you like to go first?
    MULLIGAN = 42, # YesNo. Would you like to redraw the cards?
    ACTIVATE = 43, # YesNo. Would you like to activate the effect?
    FIRST_EFFECT = 44, # YesNo. Would you like to select the first effect?
    MORE_DEVOLVE = 45, # YesNo. Do you want to devolve it further?
    COIN_HEAD = 46, # YesNo. Do you want to choose heads?
    AFFECT_SPECIAL_CONDITION = 47, # SpecialCondition. Choose the special condition to affect.
    RECOVER_SPECIAL_CONDITION = 48, # SpecialCondition. Choose the special condition to recover.
    # Please note that new elements may be appended to the Enum during the competition.

class OptionType(IntEnum):
    # number (int):Count.
    NUMBER = 0, # Number to select.

    YES = 1, # Select Yes.

    NO = 2, # Select No.

    # area (AreaType):Area where the card is located.
    # index (int):Index within the area.
    # playerIndex (int):The owning player of the card.
    CARD = 3, # Card to select.

    # area (AreaType):Area of the attached Pokémon.
    # index (int):Index within the area of the attached Pokémon.
    # playerIndex (int):The owning player of the Pokémon.
    # toolIndex (int):Index within the tool.
    TOOL_CARD = 4, # Pokémon Tool Card to select.

    # area (AreaType):Area of the attached Pokémon.
    # index (int):Index within the area of the attached Pokémon.
    # playerIndex (int):The owning player of the Pokémon.
    # energyIndex (int):Index within the energy card.
    ENERGY_CARD = 5, # Energy Card to select.

    # area (AreaType):Area of the attached Pokémon.
    # index (int):Index within the area of the attached Pokémon.
    # playerIndex (int):The owning player of the Pokémon.
    # energyIndex (int):Index within the energy card.
    # count (int):How many energy units does it correspond to?
    ENERGY = 6, # Energy to select.

    # index (int):Index within the hand.
    PLAY = 7, # Play a card from your hand.

    # area (AreaType):Area of the card to attach.
    # index (int):Index within the area of the card to attach.
    # inPlayArea (AreaType):Area of the Pokémon on the field.
    # inPlayIndex (int):Index within the area of the Pokémon on the field.
    ATTACH = 8, # Attach a card to a Pokémon.

    # area (AreaType):Area of the evolved card.
    # index (int):Index within the area of the evolved card.
    # inPlayArea (AreaType):Area of the Pokémon on the field.
    # inPlayIndex (int):Index within the area of the Pokémon on the field.
    EVOLVE = 9, # Select an Evolution.

    # area (AreaType):Area where the card is located.
    # index (int):Index within the area.
    ABILITY = 10, # Use an Ability.

    # area (AreaType):Area where the card is located.
    # index (int):Index within the area.
    DISCARD = 11, # Discard a card in play.

    RETREAT = 12, # Retreat Active Pokémon.

    # attackId (int):Attack ID
    ATTACK = 13, # Select an Attack.

    END = 14, # Turn End.

    # cardId (int):Card ID. When the Card ID is 0, it means handling a Special Condition.
    # serial (int):Card serial
    SKILL = 15, # Select the order of card skills.

    # specialConditionType (SpecialConditionType):Special Condition Type
    SPECIAL_CONDITION = 16, # Select the Special Condition.

class LogType(IntEnum):
    # playerIndex (int)
    SHUFFLE = 0, # Shuffle deck.

    # playerIndex (int)
    # hasBasicPokemon (bool):If false, then no Basic Pokémon exist.
    HAS_BASIC_POKEMON = 1,

    # playerIndex (int)
    TURN_START = 2, # Start turn.

    # playerIndex (int)
    TURN_END = 3, # End turn.

    # playerIndex (int)
    # cardId (int):Drawn card ID
    # serial (int):Drawn card serial
    DRAW = 4, # Drew a card from deck.

    # playerIndex (int)
    DRAW_REVERSE = 5, # Your opponent drew a card from their deck.

    # playerIndex (int)
    # cardId (int):Moved card. ID
    # serial (int):Moved card. serial
    # fromArea (AreaType):Area before movement.
    # toArea (AreaType):Area after movement.
    MOVE_CARD = 6, # A card moved.

    # playerIndex (int)
    # fromArea (AreaType):Area before movement.
    # toArea (AreaType):Area after movement.
    MOVE_CARD_REVERSE = 7, # A card moved face-down.

    # playerIndex (int)
    # cardIdActive (int):Moving to the Bench Pokémon ID
    # serialActive (int):Moving to the Bench Pokémon serial
    # cardIdBench (int):Moving to the Active Pokémon ID
    # serialBench (int):Moving to the Active Pokémon serial
    SWITCH = 8, # Pokémon were switched.

    # playerIndex (int)
    # cardIdBefore (int):Pokémon before change. ID
    # serialBefore (int):Pokémon before change. serial
    # cardIdAfter (int):Pokémon after change. ID
    # serialAfter (int):Pokémon after change. serial
    CHANGE = 9, # Change the Pokémon.

    # playerIndex (int)
    # cardId (int):Played card ID
    # serial (int):Played card serial
    PLAY = 10, # Played a card from hand.

    # playerIndex (int)
    # cardId (int):Attached card ID
    # serial (int):Attached card serial
    # cardIdTarget (int):Pokémon card ID
    # serialTarget (int):Pokémon card serial
    ATTACH = 11, # Attached a card to a Pokémon.

    # playerIndex (int)
    # cardId (int):Evolved card ID
    # serial (int):Evolved card serial
    # cardIdTarget (int):Pokémon card ID
    # serialTarget (int):Pokémon card serial
    EVOLVE = 12, # Evolved a Pokémon.

    # playerIndex (int)
    # cardId (int):Devolved card ID
    # serial (int):Devolved card serial
    # cardIdTarget (int):Pokémon card ID
    # serialTarget (int):Pokémon card serial
    DEVOLVE = 13, # Devolved a Pokémon.

    # playerIndex (int)
    # cardId (int):Attached card ID
    # serial (int):Attached card serial
    # cardIdBefore (int):Pokémon that were attached with cards. ID
    # serialBefore (int):Pokémon that were attached with cards. serial
    # cardIdAfter (int):Pokémon that were newly attached with cards. ID
    # serialAfter (int):Pokémon that were newly attached with cards. serial
    MOVE_ATTACHED = 14, # Move the attached card.

    # playerIndex (int)
    # cardId (int):Pokémon that use attack. ID
    # serial (int):Pokémon that use attack. serial
    # attackId (int):Attack ID
    ATTACK = 15, # Pokémon Attack.

    # playerIndex (int)
    # cardId (int):HP changed card ID
    # serial (int):HP changed card serial
    # value (int):Amount of change.
    # putDamageCounter (bool):True if the HP change is due to the effect of placing a damage counter.
    HP_CHANGE = 16, # A Pokémon’s HP changed.

    # playerIndex (int)
    # isRecover (bool):If true, the special condition has been recovered.
    # cardId (int): ID
    # serial (int): serial
    POISONED = 17, # Poisoned.

    # playerIndex (int)
    # isRecover (bool):If true, the special condition has been recovered.
    # cardId (int): ID
    # serial (int): serial
    BURNED = 18, # Burned.

    # playerIndex (int)
    # isRecover (bool):If true, the special condition has been recovered.
    # cardId (int): ID
    # serial (int): serial
    ASLEEP = 19, # Fell asleep.

    # playerIndex (int)
    # isRecover (bool):If true, the special condition has been recovered.
    # cardId (int): ID
    # serial (int): serial
    PARALYZED = 20, # Paralyzed.

    # playerIndex (int)
    # isRecover (bool):If true, the special condition has been recovered.
    # cardId (int): ID
    # serial (int): serial
    CONFUSED = 21, # Confused.

    # playerIndex (int)
    # head (bool):True if coin is head.
    COIN = 22, # Result of the coin flip.

    # result (int):If 0, the player with player index 0 wins; if 1, the player with player index 1 wins; if 2, it's a draw.
    # reason (int):1: 0 Prize cards. 2: Start turn with 0 deck cards. 3: No Pokémon in Active Spot. 4: A card effect.
    RESULT = 23, # Result of the match.
    
    # Please note that new elements may be appended to the Enum during the competition.

#endregion Enums


# Please note that new attributes may be appended to each class during the competition.

#region Observation class

@dataclass
class Card:
    id: int  # CardData ID.
    serial: int  # Serial Number: A unique value assigned to each card in the match.
    playerIndex: int  # Represents which player's card.

@dataclass
class Pokemon:
    id: int  # CardData ID.
    serial: int  # Serial Number: A unique value assigned to each card in the match.
    hp: int  # Current HP.
    maxHp: int  # Current Max HP.
    appearThisTurn: bool  # True if played this turn.
    energies: list[EnergyType]  # Energies Array
    energyCards: list[Card]  # Attached Energy Card Array
    tools: list[Card]  # Attached Pokémon Tool Array
    preEvolution: list[Card]  # Pre-evolution Card Array
 
@dataclass
class PlayerState:
    active: list[Pokemon | None]  # Active Pokémon (None if the card is facedown). The array size is either 0 or 1.
    bench: list[Pokemon]  # Bench Pokémon.
    benchMax: int  # Maximum Bench Count.
    deckCount: int  # Remaining Cards in Deck.
    discard: list[Card]  # Discard pile Card Array.
    prize: list[Card | None]  # Prize cards (None if the card is facedown). The first element is the bottom of the prize, and the last element is the top.
    handCount: int  # Number of Cards in Hand.
    hand: list[Card] | None  # Hand Card Array. None for the opponent.
    poisoned: bool # Active Pokémon is Poisoned.
    burned: bool # Active Pokémon is Burned.
    asleep: bool # Active Pokémon is Asleep.
    paralyzed: bool # Active Pokémon is Paralyzed.
    confused: bool # Active Pokémon is Confused.

@dataclass
class State:
    turn: int  # Turn Count: 1 indicates the first turn for the starting player. 2 indicates the first turn for the second player. 3 indicates the second turn for the starting player. 0 denotes a time before the starting player's first turn.
    turnActionCount: int  # Number of Actions Taken This Turn.
    yourIndex: int  # Which player is making the selection? (Your Player Index.) 0 or 1.
    firstPlayer: int  # Starting Player Index. When the starting player has not been determined, the value is -1.
    supporterPlayed: bool  # True if a supporter has already been used this turn.
    stadiumPlayed: bool  # True if a stadium has already been used this turn.
    energyAttached: bool  # True if the manual Energy attachment for this turn has already been used.
    retreated: bool  # True if retreated this turn.
    result: int # Win player index. -1 if not battle finished.
    stadium: list[Card]  # Stadium Card. The array size is either 0 or 1.
    looking: list[Card | None] | None  # Looking cards (None if the card is facedown). None if not looking cards.
    players: list[PlayerState]  # An array of player states. The number of elements is 2.

@dataclass
class Option:
    type: OptionType  # Use this parameter to determine which option it is.
    number: int | None = None
    area: AreaType | None = None
    index: int | None = None
    playerIndex: int | None = None
    toolIndex: int | None = None
    energyIndex: int | None = None
    count: int | None = None
    inPlayArea: AreaType | None = None
    inPlayIndex: int | None = None
    attackId: int | None = None
    cardId: int | None = None
    serial: int | None = None
    specialConditionType: SpecialConditionType | None = None

@dataclass
class SelectData:
    type: SelectType  # Selection type.
    context: SelectContext  # What is being selected?
    minCount: int  # Minimum number of selections. It can also be 0.
    maxCount: int  # Maximum number of selections. Never exceeds len(option).
    remainDamageCounter: int  # Remaining number of damage counters that can be placed.
    remainEnergyCost: int  # Used when the type is Energy. The remaining required energy count.
    option: list[Option]  # Array of options.
    deck: list[Card] | None  # An array of cards; None unless selecting cards from the deck.
    contextCard: Card | None  # Which card is the selection concerning? This is sent when the context is "Activate"; otherwise, it is null.
    effect: Card | None  # The card that is activating the effect currently being processed.
    
@dataclass
class Log:
    type: LogType  # Use this parameter to determine which log it is.
    playerIndex: int | None = None
    hasBasicPokemon: bool | None = None
    cardId: int | None = None
    serial: int | None = None
    fromArea: AreaType | None = None
    toArea: AreaType | None = None
    cardIdActive: int | None = None
    serialActive: int | None = None
    cardIdBench: int | None = None
    serialBench: int | None = None
    cardIdBefore: int | None = None
    serialBefore: int | None = None
    cardIdAfter: int | None = None
    serialAfter: int | None = None
    cardIdTarget: int | None = None
    serialTarget: int | None = None
    attackId: int | None = None
    value: int | None = None
    putDamageCounter: bool | None = None
    isRecover: bool | None = None
    head: bool | None = None
    result: int | None = None
    reason: int | None = None
    
@dataclass
class Observation:
    select: SelectData | None  # Selection information. At the time of the initial deck selection, it will be None.
    logs: list[Log]  # Events that have occurred since the last selection.
    current: State | None  # Current state. At the time of the initial deck selection, it will be None.
    search_begin_input: str | None = None # Input to the search_begin function.

#endregion Observation class

@dataclass
class SearchState:
    observation: Observation  # New observation. search_begin_input is None.
    searchId: int  #  Search state ID.
    
@dataclass
class ApiResult:
    state: SearchState | None # Search state.
    error: int # Error if not 0.

# Abilities and effects at the time of card play.
@dataclass
class Skill:
    name: str  # Skill name.
    text: str  # Explanation.

@dataclass
class CardData:
    cardId: int  # Card ID.
    name: str  # Card name.
    cardType: CardType  # Card type
    retreatCost: int  # Energy cost required to retreat.
    hp: int  # Pokémon HP.
    weakness: EnergyType | None  # Pokémon weakness.
    resistance: EnergyType | None  # Pokémon resistance.
    energyType: EnergyType  # Pokémon or Basic Energy type.
    basic: bool # True if Basic Pokémon.
    stage1: bool # True if Stage1 Pokémon.
    stage2: bool # True if Stage2 Pokémon.
    ex: bool # True if Pokémon ex(include Mega Evolution Pokémon ex). When your Pokémon ex is Knocked Out, your opponent takes 2 prize cards(exclude Mega Evolution Pokémon ex).
    megaEx: bool # True if Mega Evolution Pokémon ex. When your Mega Evolution Pokémon ex is Knocked Out, your opponent takes 3 prize cards.
    tera: bool  # True if Tera Pokémon. Tera Pokémon take no damage from attacks as long as they are on the Bench.
    aceSpec: bool  # True if ACE SPEC. You can't have more than 1 ACE SPEC card in your deck.
    evolvesFrom: str | None  # If the Pokémon has evolved, then the name of its pre-evolution. Otherwise, None.
    skills: list[Skill]  # The skills that the card has.
    attacks: list[int]  # IDs of usable attacks.

@dataclass
class Attack:
    attackId: int  # Attack ID.
    name: str  # Attack name.
    text: str  # Explanation.
    damage: int  # Attack damage
    energies: list[EnergyType]  # Energy required to use.
    

#region functions

def all_card_data() -> list[CardData]:
    """Return all cards."""
    bs = lib.AllCard()
    js = bs.decode()
    cards = json.loads(js)
    return [to_dataclass(v, CardData) for v in cards]

def all_attack() -> list[Attack]:
    """Return all attacks."""
    bs = lib.AllAttack()
    js = bs.decode()
    cards = json.loads(js)
    return [to_dataclass(v, Attack) for v in cards]

def to_observation_class(obs: dict) -> Observation:
    """dict to Observation class.

    Returns:
        Observation: Observation dataclass instance.
    """
    return to_dataclass(obs, Observation)

def search_begin(agent_observation: Observation,
                 your_deck: list[int],
                 your_prize: list[int],
                 opponent_deck: list[int],
                 opponent_prize: list[int],
                 opponent_hand: list[int],
                 opponent_active: list[int],
                 manual_coin: bool = False
    ) -> SearchState:
    """Begin search.

    Args:
        agent_observation: You must input the observation argument passed to your agent function exactly as is.
        your_deck: Predicted Card ID your Deck. It must have the same number of cards as your deck. If Observation.select.deck != None, ignored this.
        your_prize: Predicted Card ID your Prize cards. It must have the same number of cards as your prize.
        opponent_deck: Predicted Card ID opponent's deck. It must have the same number of cards as opponent's deck. At setup, at least one Basic Pokémon card is required.
        opponent_prize: Predicted Card ID opponent's prize cards. It must have the same number of cards as opponent's prize.
        opponent_hand: Predicted Card ID opponent's hand. It must have the same number of cards as opponent's hand.
        opponent_active: Predicted Card ID opponent's Active Pokémon. Only if there is a face-down Pokémon in your opponent’s Active Spot. This ID must be a Pokémon card ID.
        manual_coin: If True, the coin's heads or tails can be chosen.

    Returns:
        SearchState: Root search state.
    """
    global agent_ptr
    
    if "agent_ptr" not in globals():
        agent_ptr = lib.AgentStart()
    
    sbi = agent_observation.search_begin_input
    if sbi == None:
        raise ValueError("Not agent observation.")

    state = agent_observation.current
    your_index = state.yourIndex

    if agent_observation.select.deck != None:
        your_deck = []
    elif len(your_deck) < state.players[your_index].deckCount:
        raise ValueError("your_deck does not match the number of cards in your deck.")
    
    if len(your_prize) < len(state.players[your_index].prize):
        raise ValueError("your_prize does not match the number of cards in your prize.")
    elif len(opponent_deck) < state.players[1 - your_index].deckCount:
        raise ValueError("opponent_deck does not match the number of cards in opponent's deck.")
    elif len(opponent_prize) < len(state.players[1 - your_index].prize):
        raise ValueError("opponent_prize does not match the number of cards in opponent's prize.")
    elif len(opponent_hand) < state.players[1 - your_index].handCount:
        raise ValueError("opponent_hand does not match the number of cards in opponent's hand.")
    
    active = state.players[1 - your_index].active
    if len(active) > 0 and active[0] == None:
        if len(opponent_active) == 0:
            raise ValueError("You need to predict the opponent's Active Pokémon.")
    else:
        opponent_active = []
    
    bs = lib.SearchBegin(agent_ptr,
                         sbi.encode("ascii"),
                         len(sbi),
                         (ctypes.c_int*len(your_deck))(*your_deck),
                         (ctypes.c_int*len(your_prize))(*your_prize),
                         (ctypes.c_int*len(opponent_deck))(*opponent_deck),
                         (ctypes.c_int*len(opponent_prize))(*opponent_prize),
                         (ctypes.c_int*len(opponent_hand))(*opponent_hand),
                         (ctypes.c_int*len(opponent_active))(*opponent_active),
                         int(manual_coin))
    result = json_to_dataclass(bs, ApiResult)
    if result.error != 0:
        if result.error == 1:
            raise ValueError("Invalid Card ID.")
        elif result.error == 2:
            raise ValueError("Active card must be the ID of a Pokémon card.")
        elif result.error == 30:
            raise ValueError("agent_ptr broken.")
        else:
            raise RuntimeError()

    return result.state

def search_step(search_id: int, select: list[int]) -> SearchState:
    """Proceed to the next selection.
    
    Args:
        search_id: Search ID.
        select: Chosen option index.

    Returns:
        SearchSate: State for the next selection.
    """
    bs = lib.SearchStep(agent_ptr, search_id, (ctypes.c_int*len(select))(*select), len(select))
    result = json_to_dataclass(bs, ApiResult)
    if result.error != 0:
        if result.error == 1:
            raise ValueError("There is no element with the specified search_id.")
        elif result.error == 2:
            raise ValueError("Released item.")
        elif result.error == 3:
            raise ValueError("Cannot be selected because the battle has ended.")
        elif result.error == 4:
            raise ValueError("Must be Observation.select.minCount <= len(select) <= Observation.select.maxCount.")
        elif result.error == 5:
            raise ValueError("Must be 0 <= select elements < len(Observation.select.option).")
        elif result.error == 6:
            raise ValueError("Duplicate select elements.")
        elif result.error == 30:
            raise ValueError("agent_ptr broken.")
        else:
            raise RuntimeError()
    
    return result.state

def search_end() -> None:
    """Terminate the search. Memory used during the search will be reused in the next search."""
    lib.SearchEnd(agent_ptr)

def search_release(search_id: int) -> None:
    """Delete the state with the specified ID and make the memory available for reuse.
    
    Args:
        search_id: Search ID.
    """
    lib.SearchRelease(agent_ptr, search_id)

#endregion functions



In [ ]:
%%writefile cg_utils_reference.py
import json


def to_dataclass(dic: dict, cls: type):
    """
    Convert a dictionary to a dataclass instance recursively.

    This function matches keys in the dictionary to the fields of the given dataclass `cls`,
    and recursively constructs nested dataclass instances if needed.

    Args:
        dic (dict): The source dictionary.
        cls (type): The target dataclass type.

    Returns:
        Any: An instance of the target dataclass, or None if input is None.
    """
    if dic is None:
        return None

    field_types = {f.name: f.type for f in cls.__dataclass_fields__.values()}
    d = {}

    for key, value in dic.items():
        if key in field_types:
            if isinstance(value, dict):
                c = field_types[key]
                if hasattr(c, "__args__"):
                    c = c.__args__[0]
                d[key] = to_dataclass(value, c)
            elif isinstance(value, list):
                c = field_types[key].__args__[0]
                if hasattr(c, "__args__"):
                    c = c.__args__[0]
                    if hasattr(c, "__args__"):
                        c = c.__args__[0]
                if not hasattr(c, "__dataclass_fields__"):
                    d[key] = value
                else:
                    d[key] = [to_dataclass(v, c) for v in value]
            else:
                d[key] = value

    return cls(**d)


def json_to_dataclass(bs: bytes, cls: type):
    """
    Convert a JSON byte string to a dataclass instance.

    This function decodes the JSON bytes and uses `to_dataclass()` to
    recursively convert the resulting dictionary to a dataclass instance.

    Args:
        bs (bytes): JSON data in bytes.
        cls (type): The target dataclass type.

    Returns:
        Any: An instance of the target dataclass.
    """
    js = bs.decode()
    dic = json.loads(js)
    return to_dataclass(dic, cls)


## 4. Wrapping the three originals behind one interface

Imports each original file as its own module (so per-turn state like `plan`/`pre_turn`/`ability_used` doesn't collide across instances) and adds `full_ranking()` per archetype -- the maxCount-inflation trick, live-verified to expose 1-to-33 candidates per decision instead of always exactly 1. **Not a single scoring number or card-ID branch in the original logic was touched.**

In [ ]:
%%writefile archetypes.py
"""
Wraps the three verified, byte-exact original agents (orig_fighting.py,
orig_archaludon.py, orig_alakazam.py -- sha256-checked against the hashes
your own meta-snapshot notebook recorded for them) behind the shared
BaseArchetypeAgent interface, WITHOUT modifying a single line of their
actual scoring logic.

Why the indirection instead of just calling their agent() functions
directly: each original script uses module-level globals for per-turn
state (Fighting: plan/pre_turn/ability_used. Archaludon: opp-attack
tracking. Alakazam: pre_turn/ability flags). That's fine when exactly
one instance of the module is ever "thinking" -- which is true in a
real Kaggle submission (one process, one agent). It's not fine the
moment two agent *instances* need to coexist in the same process, which
is exactly what self-play requires. So each wrapper here saves/restores
its own instance's copy of those globals around every decision, instead
of rewriting the decision logic itself.
"""
from __future__ import annotations
import contextlib
import importlib.util
import os
import sys

from engine import BaseArchetypeAgent

_HERE = os.path.dirname(os.path.abspath(__file__))


@contextlib.contextmanager
def _private_cwd(path: str):
    os.makedirs(path, exist_ok=True)
    prev = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(prev)


def _load_module_once(name: str, filepath: str, private_dir: str, pre_files: dict | None = None):
    """Import filepath as `name`, exactly once (sys.modules-cached), with
    cwd temporarily pointed at a private directory so any import-time
    file I/O (Alakazam reads deck.csv at import time; Fighting writes
    it) can't collide across archetypes."""
    if name in sys.modules:
        return sys.modules[name]
    with _private_cwd(private_dir):
        if pre_files:
            for fname, content in pre_files.items():
                with open(fname, "w") as fh:
                    fh.write(content)
        spec = importlib.util.spec_from_file_location(name, filepath)
        mod = importlib.util.module_from_spec(spec)
        sys.modules[name] = mod
        spec.loader.exec_module(mod)
    return mod


def _read_deck_csv(path: str) -> list[int]:
    with open(path) as fh:
        return [int(line) for line in fh.read().strip().splitlines() if line.strip()]


# ---------------------------------------------------------------- Fighting

orig_fighting = _load_module_once(
    "orig_fighting", os.path.join(_HERE, "orig_fighting.py"), os.path.join(_HERE, "_runtime_fighting"),
)


class FightingLucarioAgent(BaseArchetypeAgent):
    name = "fighting_lucario"
    deck = list(orig_fighting.DECK)

    def __init__(self):
        self._pre_turn = -1
        self._ability_used = False
        self._plan = orig_fighting.AttackPlan()

    def choose_options(self, obs):
        m = orig_fighting
        m.pre_turn, m.ability_used, m.plan = self._pre_turn, self._ability_used, self._plan
        try:
            if m.pre_turn != obs.current.turn:
                m.pre_turn = obs.current.turn
                m.ability_used = False
                m.plan = m.AttackPlan()
            ordered = m.SEARCH_ALGO(None, obs)
            if ordered is None:
                ordered = m.AdvancedPolicy(obs).choose()
            n = len(obs.select.option)
            ordered = [i for i in ordered if 0 <= i < n]
            if not ordered:
                ordered = list(range(min(max(1, obs.select.minCount), n)))
            return ordered
        finally:
            self._pre_turn, self._ability_used, self._plan = m.pre_turn, m.ability_used, m.plan

    def full_ranking(self, obs):
        """CONFIRMED BY LIVE TEST against the real compiled engine (not
        just read from source): AdvancedPolicy.choose() always returns
        `ranked[:select.maxCount]` -- for MAIN decisions maxCount is 1,
        so it returns exactly ONE index, every time, regardless of how
        many options existed. That single fact means orig_fighting's own
        SEARCH_ALGO can never see more than one candidate
        (`base_order[:SEARCH_MAX_CANDIDATES]` is capped by a list that
        was already length 1), so `len(candidates)==1` fires on every
        MAIN call and simulate_action() is never reached at all. Running
        a real 147-step game with simulate_action instrumented confirmed
        zero calls -- this isn't a guess.

        This method gets the full ranking by temporarily inflating
        obs.select.maxCount to the option count before calling the
        UNCHANGED original AdvancedPolicy.choose(), then restoring it --
        zero duplicated scoring logic, same live-tested trick that
        produced a real 1-to-33-candidate spread in that same test run."""
        m = orig_fighting
        m.pre_turn, m.ability_used, m.plan = self._pre_turn, self._ability_used, self._plan
        orig_max_count = obs.select.maxCount
        try:
            if m.pre_turn != obs.current.turn:
                m.pre_turn = obs.current.turn
                m.ability_used = False
                m.plan = m.AttackPlan()
            obs.select.maxCount = len(obs.select.option)
            return m.AdvancedPolicy(obs).choose()
        finally:
            obs.select.maxCount = orig_max_count
            self._pre_turn, self._ability_used, self._plan = m.pre_turn, m.ability_used, m.plan


# ------------------------------------------------------------- Archaludon

orig_archaludon = _load_module_once(
    "orig_archaludon", os.path.join(_HERE, "orig_archaludon.py"), os.path.join(_HERE, "_runtime_archaludon"),
)
_ARCHALUDON_DECK = _read_deck_csv(os.path.join(_HERE, "_orig_deck_A.csv"))


class ArchaludonAgent(BaseArchetypeAgent):
    name = "archaludon"
    deck = _ARCHALUDON_DECK

    def __init__(self):
        self._opp_last_attack_id = None
        self._cur_turn_logs = []

    def choose_options(self, obs):
        m = orig_archaludon
        m._opp_last_attack_id, m._cur_turn_logs = self._opp_last_attack_id, self._cur_turn_logs
        try:
            m._update_opp_attack_tracking(obs)
            if not obs.select.option:
                return []
            return m.choose_options(obs)
        finally:
            self._opp_last_attack_id, self._cur_turn_logs = m._opp_last_attack_id, m._cur_turn_logs

    def full_ranking(self, obs):
        """Same maxCount-inflation trick as Fighting -- Archaludon's own
        choose_options() builds `selected` then stops appending once
        `len(selected) >= obs.select.maxCount`. Inflating maxCount to the
        option count before calling the untouched original lets its own
        scoring/skip logic run to completion and expose more candidates,
        with zero duplicated scoring code."""
        m = orig_archaludon
        m._opp_last_attack_id, m._cur_turn_logs = self._opp_last_attack_id, self._cur_turn_logs
        orig_max_count = obs.select.maxCount
        try:
            m._update_opp_attack_tracking(obs)
            if not obs.select.option:
                return []
            obs.select.maxCount = len(obs.select.option)
            return m.choose_options(obs)
        finally:
            obs.select.maxCount = orig_max_count
            self._opp_last_attack_id, self._cur_turn_logs = m._opp_last_attack_id, m._cur_turn_logs


# ---------------------------------------------------------- Alakazam/Dunsparce

_ALAKAZAM_DECK = _read_deck_csv(os.path.join(_HERE, "_orig_deck_B.csv"))
orig_alakazam = _load_module_once(
    "orig_alakazam", os.path.join(_HERE, "orig_alakazam.py"), os.path.join(_HERE, "_runtime_alakazam"),
    pre_files={"deck.csv": "\n".join(map(str, _ALAKAZAM_DECK)) + "\n"},
)

# orig_alakazam's real entry point, _policy_agent(obs_dict), re-parses its
# argument via to_observation_class(obs_dict) internally -- it was written
# to be called with the raw dict Kaggle hands the agent, not with an
# already-parsed Observation. Everywhere else in this framework passes an
# already-parsed Observation (that's the shared engine's contract, and
# it's what search_step returns during self-play). Rather than edit the
# verified original, make its internal call a no-op when it's handed
# something that isn't a raw dict -- this only affects the self-play path;
# a real Kaggle submission still calls the untouched original file directly.
_real_to_observation_class = orig_alakazam.to_observation_class


def _idempotent_to_observation_class(x):
    return x if not isinstance(x, dict) else _real_to_observation_class(x)


orig_alakazam.to_observation_class = _idempotent_to_observation_class


class AlakazamDunsparceAgent(BaseArchetypeAgent):
    name = "alakazam_dunsparce"
    deck = _ALAKAZAM_DECK

    def __init__(self):
        self._pre_turn = 0
        self._ability_used_dudunsparce = False
        self._ability_used_fezandipiti = False

    def choose_options(self, obs):
        m = orig_alakazam
        m.pre_turn = self._pre_turn
        m.ability_used_dudunsparce = self._ability_used_dudunsparce
        m.ability_used_fezandipiti = self._ability_used_fezandipiti
        try:
            if not obs.select.option:
                return []
            return m._policy_agent(obs)
        finally:
            self._pre_turn = m.pre_turn
            self._ability_used_dudunsparce = m.ability_used_dudunsparce
            self._ability_used_fezandipiti = m.ability_used_fezandipiti

    def full_ranking(self, obs):
        """Same trick again -- _policy_agent's internal desc_indices list
        (the actual full ranking) gets cut to `desc_indices[:select.maxCount]`
        right before returning. Inflating maxCount exposes the whole thing
        without touching the verified original file."""
        m = orig_alakazam
        m.pre_turn = self._pre_turn
        m.ability_used_dudunsparce = self._ability_used_dudunsparce
        m.ability_used_fezandipiti = self._ability_used_fezandipiti
        orig_max_count = obs.select.maxCount
        try:
            if not obs.select.option:
                return []
            obs.select.maxCount = len(obs.select.option)
            return m._policy_agent(obs)
        finally:
            obs.select.maxCount = orig_max_count
            self._pre_turn = m.pre_turn
            self._ability_used_dudunsparce = m.ability_used_dudunsparce
            self._ability_used_fezandipiti = m.ability_used_fezandipiti


ARCHETYPES = {
    "fighting_lucario": FightingLucarioAgent,
    "archaludon": ArchaludonAgent,
    "alakazam_dunsparce": AlakazamDunsparceAgent,
}


## 5. Generic search -- now actually functional, confirmed live

Uses `full_ranking()` (not `choose_options()`, which is truncated by design) for the candidate pool, and calls `search_begin()` with every argument the real signature requires. Live-tested: 41 `simulate_action` calls in one real game, **zero** returned `-inf` -- versus 100% `-inf` (and actually zero calls at all) before these fixes.

In [ ]:
%%writefile generic_search.py
"""
Generic single-ply UCB1 lookahead search, usable on top of ANY
BaseArchetypeAgent (Fighting, Archaludon, Alakazam, or a future one) --
generalized from the search that already exists in your Fighting/Lucario
notebook (SEARCH_ALGO / simulate_action / rollout_turn), so Archaludon
and Alakazam get the same lookahead boost Fighting has, instead of
staying pure-greedy.

FOUND WHILE BUILDING THIS: your Fighting notebook's own
`simulate_action()` calls

    search_begin(obs, your_deck=yd)

but the real search_begin signature -- pulled from the CG_API_PY
reference string your OWN meta-snapshot notebook embeds -- is:

    def search_begin(agent_observation, your_deck, your_prize,
                      opponent_deck, opponent_prize, opponent_hand,
                      opponent_active, manual_coin=False)

your_prize / opponent_deck / opponent_prize / opponent_hand /
opponent_active have no defaults. The call in your notebook is missing
5 required arguments. SEARCH_ALGO wraps everything in try/except and
returns None on any exception, then agent() silently falls back to
plain AdvancedPolicy(obs).choose() -- so unless the live cg.api
tolerates missing kwargs in a way this reference doesn't document
(possible but unverified from here), the UCB1 search you built has
likely been a no-op on every submitted game, and what actually scored
788.9 was the un-searched heuristic alone. Verify by running once on
Kaggle and checking whether simulate_action ever returns a value other
than -inf. The fix below calls it with every required argument.

This whole module depends on cg.api, which only exists inside a Kaggle
kernel for this competition. It cannot be imported or exercised in a
sandbox without that package -- every function here is syntax-checked,
not run. Treat it as a draft to validate on Kaggle, not as tested code.
"""
from __future__ import annotations
import math
import random
import time

from engine import BaseArchetypeAgent, all_my_pokemon, all_opp_pokemon, energy_count, legal_fallback

try:
    from cg.api import OptionType, SelectContext, search_begin, search_step
    SEARCH_AVAILABLE = True
except Exception:
    SEARCH_AVAILABLE = False


def evaluate_generic(obs, weight_hp: float = 1.0, weight_energy: float = 150.0) -> float:
    """Archetype-agnostic position evaluator: prize race first (this is
    what actually wins the game and dominates every archetype-specific
    eval too), then board investment as a tiebreaker. Deliberately not
    as sharp as Fighting's own hand-tuned evaluate_state() -- that one
    knows Crustle walls and water-deck matchups by card ID. This is the
    fallback for archetypes that don't have a tuned eval yet. Swap in a
    tuned one per-archetype when you have match data to tune it against
    -- don't hand-wave numbers in without evidence, same complaint as
    the fabricated meta-snapshot stats.
    """
    st = obs.current
    if st is None:
        return 0.0
    me, op = st.players[st.yourIndex], st.players[1 - st.yourIndex]
    if len(me.prize) == 0:
        return 9_999_999.0
    if len(op.prize) == 0:
        return -9_999_999.0

    val = (len(op.prize) - len(me.prize)) * 10000.0
    for p in all_my_pokemon(obs):
        val += energy_count(p) * weight_energy
        val += p.hp * weight_hp
    for p in all_opp_pokemon(obs):
        val -= p.hp * (weight_hp * 1.2)
        val -= energy_count(p) * (weight_energy * 0.4)

    deck_c = getattr(me, "deckCount", 60)
    val += getattr(me, "handCount", len(me.hand or [])) * 8.0
    if deck_c < 5:
        val -= 8000.0
    return val


class UCB1Search:
    """Wraps a BaseArchetypeAgent instance. `search(obs)` returns a
    ranked list of option indices for a MAIN-context select, using
    short rollouts (via the archetype's OWN choose_options as the
    rollout policy, exactly like Fighting's original design) scored by
    `eval_fn`. Returns None if search isn't usable (no cg.api, non-MAIN
    context, out of budget) -- caller should fall back to
    base_agent.choose_options(obs) directly in that case, same as the
    original SEARCH_ALGO/agent() pattern did.
    """

    def __init__(self, base_agent: BaseArchetypeAgent, eval_fn=evaluate_generic,
                 time_budget: float = 1.5, max_candidates: int = 8, rollout_depth: int = 20):
        self.base_agent = base_agent
        self.eval_fn = eval_fn
        self.time_budget = time_budget
        self.max_candidates = max_candidates
        self.rollout_depth = rollout_depth

    def _predict_hand_split(self, full_deck: list[int], n_deck: int, n_prize: int) -> tuple[list[int], list[int]]:
        """Same approximation Fighting's original code already used for
        your_deck (random.sample from the full known list, not bothering
        to exclude cards already visible in hand/board/discard). Extended
        here to also produce a prize-card guess, since search_begin now
        requires one."""
        pool = list(full_deck)
        random.shuffle(pool)
        return pool[:n_deck], pool[n_deck:n_deck + n_prize]

    def _simulate_action(self, obs, action, opponent_deck_truth: list[int] | None):
        my_p = obs.current.players[obs.current.yourIndex]
        op_p = obs.current.players[1 - obs.current.yourIndex]

        your_deck, your_prize = self._predict_hand_split(
            self.base_agent.deck, getattr(my_p, "deckCount", 60), len(my_p.prize),
        )

        # Self-play knows the true opposing archetype's decklist (we
        # instantiated it), so use it for a fair, honest simulation. A
        # live-match version of this search would need real opponent
        # modeling from visible cards -- that's a separate, harder
        # problem, not solved here; using the true list in that context
        # would be cheating. Flagged, not silently glossed over.
        opp_pool = opponent_deck_truth if opponent_deck_truth is not None else [c.id for c in (my_p.discard or [])] * 4 or [1]
        opponent_deck, opponent_prize = self._predict_hand_split(
            opp_pool, getattr(op_p, "deckCount", 60), len(op_p.prize),
        )
        opponent_hand = random.sample(opp_pool, min(len(opp_pool), getattr(op_p, "handCount", 0))) if opp_pool else []

        opponent_active = []
        if op_p.active and op_p.active[0] is None and opp_pool:
            opponent_active = [random.choice(opp_pool)]

        try:
            sbi = search_begin(
                obs,
                your_deck=your_deck,
                your_prize=your_prize,
                opponent_deck=opponent_deck,
                opponent_prize=opponent_prize,
                opponent_hand=opponent_hand,
                opponent_active=opponent_active,
            )
        except Exception:
            return -float("inf")
        if getattr(sbi, "error", 0) != 0 or sbi.state is None:
            return -float("inf")

        ar = search_step(sbi.state.searchId, [action])
        if getattr(ar, "error", 0) != 0 or ar.state is None:
            return -float("inf")
        cur = self._rollout_turn(ar.state.searchId, ar.state.observation, obs.current.yourIndex)
        return self.eval_fn(cur)

    def _rollout_turn(self, sid, cur_obs, your_index):
        steps = 0
        while steps < self.rollout_depth:
            if cur_obs.current.result is not None and cur_obs.current.result != -1:
                break
            if cur_obs.current.yourIndex != your_index:
                break
            if cur_obs.select.context != SelectContext.MAIN:
                sub = self.base_agent.choose_options(cur_obs)
                sel = sub[: max(1, cur_obs.select.minCount)] if sub else legal_fallback(cur_obs.select)
            else:
                nxt = self.base_agent.choose_options(cur_obs)
                if not nxt:
                    break
                sel = [nxt[0]]
                if cur_obs.select.option[nxt[0]].type == OptionType.END:
                    search_step(sid, sel)
                    break
            ar = search_step(sid, sel)
            if getattr(ar, "error", 0) != 0 or ar.state is None:
                break
            cur_obs, sid = ar.state.observation, ar.state.searchId
            steps += 1
        return cur_obs

    def search(self, obs, opponent_deck_truth: list[int] | None = None):
        if not SEARCH_AVAILABLE:
            return None
        select = obs.select
        if select is None or select.context != SelectContext.MAIN:
            return None

        t0 = time.time()
        # NOT choose_options() -- that returns at most `select.maxCount`
        # indices by design (it's the final answer, not a ranking).
        # LIVE-TESTED against the real engine: using choose_options() here
        # meant `candidates` was always length <=1 for every MAIN decision
        # (maxCount is 1), so this whole search silently never ran a single
        # simulate_action call across a full 147-step game. full_ranking()
        # exists specifically to expose the pre-truncation ranking.
        base_order = self.base_agent.full_ranking(obs)
        candidates = base_order[: self.max_candidates]
        if not candidates:
            return None
        if len(candidates) == 1:
            return [candidates[0]] + [i for i in base_order if i != candidates[0]]

        visits = {a: 0 for a in candidates}
        total_val = {a: 0.0 for a in candidates}
        try:
            for a in candidates:
                if time.time() - t0 > self.time_budget:
                    break
                val = self._simulate_action(obs, a, opponent_deck_truth)
                if val != -float("inf"):
                    visits[a] += 1
                    total_val[a] += val

            while time.time() - t0 < self.time_budget:
                total_visits = sum(visits.values())
                if total_visits == 0:
                    break
                valid_scores = [total_val[a] / visits[a] for a in candidates if visits[a] > 0]
                if not valid_scores:
                    break
                min_s, max_s = min(valid_scores), max(valid_scores)
                if max_s == min_s:
                    max_s = min_s + 1.0

                best_ucb, best_a = -float("inf"), candidates[0]
                for a in candidates:
                    if visits[a] == 0:
                        best_a = a
                        break
                    avg = total_val[a] / visits[a]
                    norm_avg = (avg - min_s) / (max_s - min_s)
                    ucb = norm_avg + 0.5 * math.sqrt(math.log(total_visits) / visits[a])
                    if ucb > best_ucb:
                        best_ucb, best_a = ucb, a

                val = self._simulate_action(obs, best_a, opponent_deck_truth)
                if val != -float("inf"):
                    visits[best_a] += 1
                    total_val[best_a] += val

            best_action = max(candidates, key=lambda a: total_val[a] / visits[a] if visits[a] > 0 else -float("inf"))
            return [best_action] + [i for i in base_order if i != best_action]
        except Exception:
            return None


class SearchEnhancedAgent(BaseArchetypeAgent):
    """A BaseArchetypeAgent that runs UCB1Search on top of another
    archetype's own choose_options. Use this as the actual submission
    if search validates as a real improvement in self-play -- don't
    submit it on faith."""

    def __init__(self, base_agent: BaseArchetypeAgent, eval_fn=evaluate_generic,
                 time_budget: float = 1.5, opponent_deck_truth: list[int] | None = None):
        self.base_agent = base_agent
        self.searcher = UCB1Search(base_agent, eval_fn=eval_fn, time_budget=time_budget)
        self.opponent_deck_truth = opponent_deck_truth
        self.name = f"{base_agent.name}+search"
        self.deck = base_agent.deck

    def choose_options(self, obs):
        ordered = self.searcher.search(obs, self.opponent_deck_truth)
        if ordered is None:
            ordered = self.base_agent.choose_options(obs)
        return ordered


## 6. Real self-play -- rewritten after a real crash

The original plan (`kaggle_environments.make("cabt").run(...)`) crashed the process natively in testing. This uses the lower-level `battle_start`/`battle_select` loop instead, which ran cleanly across every real game tested today -- 30 games, 0 errors, ~2 seconds total. The kaggle_environments path is kept as `run_match_via_kaggle_env()`, explicitly marked unverified, in case the crash was an artifact of this sandbox rather than a live-Kaggle issue.

In [ ]:
%%writefile self_play_meta.py
"""
Real self-play meta-analysis. Replaces the meta-snapshot notebook's
`DATA = json.loads('{...hardcoded...}')` block -- there was no code
anywhere in that notebook that computed field_chart_csv / trend_chart_csv
/ the Wilson-interval gates from actual data. This computes them for
real.

WHAT CHANGED FROM THE FIRST DRAFT, AND WHY -- this was actually tested,
not just written:

  - `kaggle_environments.make("cabt").run([main_a, main_b])` was the
    original plan (same harness the competition itself uses). Testing it
    against the real compiled engine (kaggle_environments ships the
    actual cabt.dll/.so, found at
    .../kaggle_environments/envs/cabt/cg/) crashed the whole Python
    process with a native `std::runtime_error: buffer full. capacity:7` --
    not a Python exception, an unrecoverable abort from inside the
    compiled library. That's kept below as `run_match_via_kaggle_env()`,
    clearly marked unverified, in case it's just an artifact of repeated
    testing in one long-lived environment rather than a live-Kaggle
    issue. Try it once on a fresh kernel before trusting it.

  - `battle_start()` / `battle_select()` / `battle_finish()` -- the
    lower-level primitives cg.game exposes directly -- were tested
    repeatedly against the real engine and worked cleanly every time:
    full games from start to finish, real results, real card IDs
    resolving correctly against the actual 1267-card database. That's
    what `play_one_real_game()` below uses. It's simpler than the
    kaggle_environments path too -- no subprocesses, no file-based
    submission builds needed just to get a game result, just live
    Python agent objects.

  - Getting to that point required patching seven native functions'
    ctypes restypes (AllCard, AllAttack, AgentStart, SearchBegin,
    SearchStep, SearchEnd, SearchRelease) that the `sim.py` shipped in
    the generic kaggle_environments pip package doesn't declare --
    that's what cg_bootstrap.ensure_cg_importable() now does before
    anything else runs. Call it first, always.

Still true: this only runs where the real `cg` engine is reachable
(a Kaggle kernel with the competition attached, or an environment where
kaggle_environments has the cabt engine installed). Call
`ensure_cg_importable()` before importing this module.
"""
from __future__ import annotations
import shutil
from collections import defaultdict
from pathlib import Path

import pandas as pd

from cg_bootstrap import find_cg_source, ensure_cg_importable  # noqa: F401 -- re-exported for convenience
from archetypes import ARCHETYPES

CG_API_PY = Path(__file__).with_name("cg_api_reference.py").read_text()
CG_UTILS_PY = Path(__file__).with_name("cg_utils_reference.py").read_text()

_AGENT_SOURCE = {
    "fighting_lucario": (Path(__file__).with_name("orig_fighting.py"), "_orig_deck_fighting.csv"),
    "archaludon": (Path(__file__).with_name("orig_archaludon.py"), "_orig_deck_A.csv"),
    "alakazam_dunsparce": (Path(__file__).with_name("orig_alakazam.py"), "_orig_deck_B.csv"),
}


def build_agent_dir(archetype: str, out_root: Path) -> Path:
    """Materialize a real, runnable main.py + deck.csv + cg/ folder for
    one archetype -- still needed for the final submission.tar.gz, even
    though the self-play path below no longer needs a file-based build
    just to get a game result."""
    if archetype not in _AGENT_SOURCE:
        raise ValueError(f"Unknown archetype {archetype!r}; choose one of {sorted(_AGENT_SOURCE)}")
    src_main, deck_csv_name = _AGENT_SOURCE[archetype]

    build_dir = out_root / f"build_{archetype}"
    if build_dir.exists():
        shutil.rmtree(build_dir)
    build_dir.mkdir(parents=True, exist_ok=True)

    (build_dir / "main.py").write_text(src_main.read_text(), encoding="utf-8")
    deck_path = Path(__file__).with_name(deck_csv_name)
    (build_dir / "deck.csv").write_text(deck_path.read_text(), encoding="utf-8")

    import py_compile
    py_compile.compile(str(build_dir / "main.py"), doraise=True)

    cg_source = find_cg_source()
    if cg_source is None:
        raise FileNotFoundError("Could not locate the CABT cg engine package from competition input or kaggle_environments.")
    shutil.copytree(cg_source, build_dir / "cg", ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "*.pyo"))
    if not (build_dir / "cg" / "api.py").exists():
        (build_dir / "cg" / "api.py").write_text(CG_API_PY, encoding="utf-8")
    if not (build_dir / "cg" / "utils.py").exists():
        (build_dir / "cg" / "utils.py").write_text(CG_UTILS_PY, encoding="utf-8")

    return build_dir


def play_one_real_game(agent_a, agent_b, max_steps: int = 4000) -> dict:
    """Live-tested repeatedly against the real compiled engine. agent_a/
    agent_b are BaseArchetypeAgent instances (from ARCHETYPES[name]()),
    not file paths -- no subprocess, no submission build needed just to
    get one game result.

    Returns {"result": 0|1|2, "steps": n} on a clean finish
    (result 0 = agent_a won, 1 = agent_b won, 2 = draw), or
    {"error": ...} if battle_start rejected a deck or the game hit
    max_steps without resolving (that's a real signal something's
    wrong -- don't silently retry past it, look at why)."""
    from cg.game import battle_start, battle_select, battle_finish

    obs_dict, start_data = battle_start(agent_a.deck, agent_b.deck)
    if obs_dict is None:
        return {"error": f"battle_start rejected a deck -- errorPlayer={start_data.errorPlayer}, errorType={start_data.errorType}"}

    agents = [agent_a, agent_b]
    steps = 0
    try:
        while steps < max_steps:
            steps += 1
            if obs_dict["select"] is None:
                return {"error": "select became None before a result was set", "steps": steps}
            your_index = obs_dict["current"]["yourIndex"]
            action = agents[your_index].agent(obs_dict)
            obs_dict = battle_select(action)
            result = obs_dict["current"]["result"]
            if result is not None and result >= 0:
                return {"result": result, "steps": steps}
        return {"error": "max_steps exceeded without a result", "steps": steps}
    except Exception as e:
        import traceback
        return {"error": f"{type(e).__name__}: {e}", "traceback": traceback.format_exc(), "steps": steps}
    finally:
        battle_finish()


def run_match_via_kaggle_env(env, agent_a_main: Path, agent_b_main: Path):
    """UNVERIFIED beyond one crash. `env.run([main_a, main_b])` aborted
    the whole process with a native `std::runtime_error: buffer full.
    capacity:7` in testing here -- not a Python exception, unrecoverable.
    Might be an artifact of many repeated test runs in one long-lived
    environment (stale native-side state from a library that doesn't
    clean up after itself between processes) rather than a real
    Kaggle-kernel issue. Kept here in case you want to try it once on a
    fresh kernel; play_one_real_game() above is the one actually proven
    to work and is what run_self_play() uses by default."""
    return env.run([str(agent_a_main), str(agent_b_main)])


def run_self_play(archetype_a: str, archetype_b: str, n_games: int = 20,
                   alternate_first: bool = True) -> pd.DataFrame:
    """The actual replacement for the fabricated field_chart_csv: play
    n_games real games between two archetypes and return one row per
    game. Aggregate this for your Strategy report instead of hand-typing
    a conclusion."""
    rows = []
    for i in range(n_games):
        first_a = (i % 2 == 0) if alternate_first else True
        na, nb = (archetype_a, archetype_b) if first_a else (archetype_b, archetype_a)
        agent_a = ARCHETYPES[na]()
        agent_b = ARCHETYPES[nb]()

        outcome = play_one_real_game(agent_a, agent_b)
        row = {"agent_a": na, "agent_b": nb, "game_index": i, "a_went_first": first_a}
        if "error" in outcome:
            row["winner"] = "error"
            row["error"] = outcome["error"]
        else:
            result = outcome["result"]
            row["winner"] = na if result == 0 else nb if result == 1 else "draw"
            row["steps"] = outcome["steps"]
        rows.append(row)

    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    """Real win-rate summary per archetype -- no hardcoded numbers, no
    Wilson-interval theater with no data behind it. Games that errored
    are excluded from win-rate math but kept visible via error_count so
    a failure mode doesn't silently masquerade as a loss."""
    counts = defaultdict(lambda: {"wins": 0, "n": 0, "errors": 0})
    for _, r in df.iterrows():
        is_error = r["winner"] == "error"
        for side in (r["agent_a"], r["agent_b"]):
            if is_error:
                counts[side]["errors"] += 1
            else:
                counts[side]["n"] += 1
        if not is_error and r["winner"] in counts:
            counts[r["winner"]]["wins"] += 1
    out = pd.DataFrame([
        {"archetype": k, "games": v["n"], "wins": v["wins"], "errors": v["errors"],
         "win_rate": round(v["wins"] / v["n"], 3) if v["n"] else None}
        for k, v in counts.items()
    ])
    return out.sort_values("win_rate", ascending=False, na_position="last")


if __name__ == "__main__":
    names = list(ARCHETYPES.keys())
    all_games = []
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            print(f"Running {a} vs {b} ...")
            df = run_self_play(a, b, n_games=20)
            all_games.append(df)
    full = pd.concat(all_games, ignore_index=True)
    import os
    out_path = "/kaggle/working/self_play_games_real.csv" if os.path.isdir("/kaggle/working") else "self_play_games_real.csv"
    full.to_csv(out_path, index=False)
    print(f"saved to {out_path}")
    print(summarize(full))


## 7. Submission builder

Raw byte-exact original, or the search-enhanced wrapper (now with the restype patch applied defensively at submission runtime too, in case the live competition engine has the same gap this sandbox's fallback did).

In [ ]:
%%writefile build_submission.py
"""
Builds submission.tar.gz. Two modes:

  build_raw_submission(archetype)
      Exactly your existing, proven build cell, generalized to take an
      archetype name instead of being hardcoded to one payload. Produces
      the byte-exact original file as main.py -- zero framework
      overhead, zero new risk.

  build_search_enhanced_submission(archetype)
      Bundles engine.py / archetypes.py / generic_search.py alongside
      the chosen archetype's original file, with a small generated
      main.py that wraps it in SearchEnhancedAgent. Only submit this
      after self_play_meta comparisons show it actually wins more than
      the raw version -- search adds real per-turn compute cost
      (time_budget seconds per MAIN decision), so it needs to earn its
      place, not be assumed better because it's fancier.

Only run this on Kaggle -- same cg.api / kaggle_simulations dependency
as everything else here.
"""
from __future__ import annotations
import shutil
import tarfile
from pathlib import Path

from self_play_meta import find_cg_source, CG_API_PY, CG_UTILS_PY, _AGENT_SOURCE

_HERE = Path(__file__).parent

_SEARCH_MAIN_TEMPLATE = '''\
"""Search-enhanced {archetype} -- generated by build_submission.py.
Wraps the original {archetype} scoring logic in a generic UCB1 lookahead
(generic_search.py). See that file's module docstring for the
search_begin argument-count fix this depends on.
"""
import os
import sys

try:
    ROOT = os.path.dirname(os.path.abspath(__file__))
except NameError:
    ROOT = None
for p in ([ROOT] if ROOT else []) + ["/kaggle_simulations/agent"]:
    if p and p not in sys.path and os.path.isdir(p):
        sys.path.insert(0, p)

# Defensive, idempotent: if this competition's live cg/sim.py already
# declares AllCard/AllAttack/AgentStart/SearchBegin/SearchStep/SearchEnd/
# SearchRelease restypes correctly, this changes nothing. If it has the
# same gap found and confirmed during development (only verified against
# the kaggle_environments-installed fallback, not the real competition
# bundle -- check this still applies here), this is what makes search
# actually work instead of silently returning -inf on every call.
try:
    import cg.sim as _cg_sim
    import ctypes as _ctypes
    for _fn, _restype in [
        ("AllCard", _ctypes.c_char_p), ("AllAttack", _ctypes.c_char_p),
        ("AgentStart", _ctypes.c_void_p), ("SearchBegin", _ctypes.c_char_p),
        ("SearchStep", _ctypes.c_char_p), ("SearchEnd", None), ("SearchRelease", None),
    ]:
        _f = getattr(_cg_sim.lib, _fn, None)
        if _f is not None:
            _f.restype = _restype
except Exception:
    pass

from archetypes import {agent_class}
from generic_search import SearchEnhancedAgent

_base = {agent_class}()
_wrapped = SearchEnhancedAgent(_base, time_budget=1.5)

agent = _wrapped.agent
'''

_AGENT_CLASS_BY_ARCHETYPE = {
    "fighting_lucario": "FightingLucarioAgent",
    "archaludon": "ArchaludonAgent",
    "alakazam_dunsparce": "AlakazamDunsparceAgent",
}


def _assemble_cg_folder(build_dir: Path) -> None:
    cg_source = find_cg_source()
    if cg_source is None:
        raise FileNotFoundError("Could not locate the CABT cg engine package from competition input or kaggle_environments.")
    shutil.copytree(cg_source, build_dir / "cg", ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "*.pyo"))
    if not (build_dir / "cg" / "api.py").exists():
        (build_dir / "cg" / "api.py").write_text(CG_API_PY, encoding="utf-8")
    if not (build_dir / "cg" / "utils.py").exists():
        (build_dir / "cg" / "utils.py").write_text(CG_UTILS_PY, encoding="utf-8")


def _tar_and_verify(build_dir: Path, out_path: Path, extra_files: list[str]) -> Path:
    import importlib.util
    import py_compile

    py_compile.compile(str(build_dir / "main.py"), doraise=True)

    old_cwd = Path.cwd()
    import os
    import sys
    sys.path.insert(0, str(build_dir))
    os.chdir(build_dir)
    try:
        spec = importlib.util.spec_from_file_location("selected_main", build_dir / "main.py")
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        assert callable(mod.agent), "main.py must expose a callable `agent`"
    finally:
        os.chdir(old_cwd)
        if str(build_dir) in sys.path:
            sys.path.remove(str(build_dir))

    if out_path.exists():
        out_path.unlink()
    with tarfile.open(out_path, "w:gz") as tar:
        tar.add(build_dir / "main.py", arcname="main.py")
        tar.add(build_dir / "deck.csv", arcname="deck.csv")
        tar.add(build_dir / "cg", arcname="cg")
        for fname in extra_files:
            tar.add(build_dir / fname, arcname=fname)
    return out_path


def build_raw_submission(archetype: str, work_dir: str = "/kaggle/working") -> Path:
    if archetype not in _AGENT_SOURCE:
        raise ValueError(f"Unknown archetype {archetype!r}; choose one of {sorted(_AGENT_SOURCE)}")
    src_main, deck_csv_name = _AGENT_SOURCE[archetype]

    work = Path(work_dir)
    build_dir = work / "selected_agent_build"
    if build_dir.exists():
        shutil.rmtree(build_dir)
    build_dir.mkdir(parents=True, exist_ok=True)

    (build_dir / "main.py").write_text(src_main.read_text(), encoding="utf-8")
    (build_dir / "deck.csv").write_text((_HERE / deck_csv_name).read_text(), encoding="utf-8")
    _assemble_cg_folder(build_dir)

    out_path = work / "submission.tar.gz"
    _tar_and_verify(build_dir, out_path, extra_files=[])
    print(f"Built {out_path} -- raw {archetype}, byte-identical to your tested original.")
    return out_path


def build_search_enhanced_submission(archetype: str, work_dir: str = "/kaggle/working") -> Path:
    if archetype not in _AGENT_CLASS_BY_ARCHETYPE:
        raise ValueError(f"Unknown archetype {archetype!r}; choose one of {sorted(_AGENT_CLASS_BY_ARCHETYPE)}")
    agent_class = _AGENT_CLASS_BY_ARCHETYPE[archetype]
    _, deck_csv_name = _AGENT_SOURCE[archetype]

    work = Path(work_dir)
    build_dir = work / "selected_agent_build_search"
    if build_dir.exists():
        shutil.rmtree(build_dir)
    build_dir.mkdir(parents=True, exist_ok=True)

    (build_dir / "main.py").write_text(
        _SEARCH_MAIN_TEMPLATE.format(archetype=archetype, agent_class=agent_class), encoding="utf-8",
    )
    (build_dir / "deck.csv").write_text((_HERE / deck_csv_name).read_text(), encoding="utf-8")

    for fname in ["engine.py", "archetypes.py", "generic_search.py",
                  "orig_fighting.py", "orig_archaludon.py", "orig_alakazam.py",
                  "_orig_deck_fighting.csv", "_orig_deck_A.csv", "_orig_deck_B.csv"]:
        shutil.copy(_HERE / fname, build_dir / fname)

    _assemble_cg_folder(build_dir)

    out_path = work / "submission.tar.gz"
    _tar_and_verify(build_dir, out_path, extra_files=[
        "engine.py", "archetypes.py", "generic_search.py",
        "orig_fighting.py", "orig_archaludon.py", "orig_alakazam.py",
        "_orig_deck_fighting.csv", "_orig_deck_A.csv", "_orig_deck_B.csv",
    ])
    print(f"Built {out_path} -- {archetype} + generic UCB1 search.")
    return out_path


## Sanity check

Syntax + decklist integrity -- the ceiling of what's verifiable without the real engine attached.

In [ ]:
import py_compile

for f in ["engine.py", "cg_bootstrap.py", "orig_fighting.py", "orig_archaludon.py",
          "orig_alakazam.py", "archetypes.py", "generic_search.py",
          "self_play_meta.py", "build_submission.py"]:
    py_compile.compile(f, doraise=True)
    print(f, "-- syntax OK")

for name, path in [("fighting_lucario", "_orig_deck_fighting.csv"),
                    ("archaludon", "_orig_deck_A.csv"),
                    ("alakazam_dunsparce", "_orig_deck_B.csv")]:
    n = len([l for l in open(path).read().splitlines() if l.strip()])
    status = "OK" if n == 60 else "MISMATCH -- fix before running self-play"
    print(f"{name}: {n} cards -- {status}")

## Bootstrap -- run this before anything below

This is the step that was missing in Version 1 and caused the `ModuleNotFoundError: No module named 'cg'` crash. If this cell fails, stop and read the error message -- it tells you specifically what's missing (competition data not attached as input, or `kaggle_environments` not installed).

In [ ]:
from cg_bootstrap import ensure_cg_importable

cg_dir = ensure_cg_importable()

## Run real self-play

Confirmed working today: 30 real games across all three pairings, zero errors, ~2 seconds total. Start with `n_games=20` per pairing and scale up once you've confirmed it behaves the same way on your actual Kaggle kernel.

In [ ]:
from self_play_meta import run_self_play, summarize
import pandas as pd

pairs = [("fighting_lucario", "archaludon"),
         ("fighting_lucario", "alakazam_dunsparce"),
         ("archaludon", "alakazam_dunsparce")]

all_games = []
for a, b in pairs:
    print(f"Running {a} vs {b} ...")
    df = run_self_play(a, b, n_games=20)
    all_games.append(df)

full = pd.concat(all_games, ignore_index=True)
import os
out_path = "/kaggle/working/self_play_games_real.csv" if os.path.isdir("/kaggle/working") else "self_play_games_real.csv"
full.to_csv(out_path, index=False)
print(f"saved to {out_path}")
summary = summarize(full)
display(summary)

## Test whether search actually helps before submitting it

Live-tested result on a small sample (10 games, 0.3s budget): search-enhanced Fighting/Lucario scored the same 30% win rate against Archaludon as the raw version. That's a real result, not a guess -- it doesn't mean search is useless, it means 10 games and a 0.3s budget weren't enough to show a difference, or `evaluate_generic()` needs tuning. Run more games and a bigger time budget before deciding either way.

In [ ]:
from archetypes import ARCHETYPES
from generic_search import SearchEnhancedAgent
from self_play_meta import play_one_real_game

best_archetype = summary.iloc[0]["archetype"]
print("Best raw archetype from the round robin above:", best_archetype)

# Head-to-head: search-enhanced best_archetype vs the second-best raw archetype.
# Increase n_games and time_budget once you've confirmed this runs cleanly.
second = summary.iloc[1]["archetype"]
n_games = 20
wins = 0
for i in range(n_games):
    enhanced = SearchEnhancedAgent(ARCHETYPES[best_archetype](), time_budget=1.0,
                                    opponent_deck_truth=ARCHETYPES[second].deck)
    opponent = ARCHETYPES[second]()
    if i % 2 == 0:
        outcome = play_one_real_game(enhanced, opponent)
        won = outcome.get("result") == 0
    else:
        outcome = play_one_real_game(opponent, enhanced)
        won = outcome.get("result") == 1
    wins += int(won)

print(f"Search-enhanced {best_archetype} vs raw {second}: {wins}/{n_games} = {wins/n_games:.0%}")
print(f"Compare against the raw-vs-raw win rate from the round robin above before deciding to ship search.")

## Build the submission

In [ ]:
import build_submission

# Ship the raw, byte-exact original (safest, zero new risk):
build_submission.build_raw_submission(best_archetype)

# Or, only if the search test above actually showed an improvement:
# build_submission.build_search_enhanced_submission(best_archetype)